# DeepTrace v5.5 -- Kaggle single T4, after the DataParallel crash

**v5.5 changes (one root cause, three edits):**
- `USE_BOTH_GPUS = False` in Module 2. DataParallel + channels_last is what produced
  `CUDA error: an illegal memory access was encountered` inside EfficientNet's depthwise
  convs during the baseline eval. DP replication flattens the parameters, which destroys the
  NHWC memory format on every replica but the first.
- Module 8 now derives `USE_CHANNELS_LAST` from `N_GPU`, so the two settings can never
  disagree again.
- Module 10 gains a `t4x1` preset: batch 32 x accum 2 = 64 images/step, 30,000 updates.
  Identical optimiser plan to `t4x2`, so every coverage number in the notebook still holds.

# DeepTrace v5.3 -- Kaggle T4 x 2, from the run that died

v5.2 got as far as shard 100/134 of the validation build and the session went with it. The log
also contained a line that should not exist. Both are fixed here, plus the bandwidth problem
that would have made the run useless even if it had survived.

## 1. `AMP: bf16` on a Tesla T4 (the expensive one)

The last run printed, verbatim: `AMP: bf16 (Turing has no bf16 tensor cores)`. A sentence that
argues with itself. On torch 2.10 + CUDA 12.8, `torch.cuda.is_bf16_supported()` returns **True**
on a T4, because its default is `including_emulation=True` and Turing can emulate bf16 in
software with no tensor-core path. So `bf16=True` went into `TrainingArguments` and every
convolution would have gone through emulation: slower than fp32, let alone fp16. The check now
gates on compute capability (`sm_80+`), which is where bf16 tensor cores actually start, and
prints what torch claimed alongside what we picked.

## 2. What killed the session: streams nobody owned

webdataset's `pipe:` reader spawns curl and only reaps it when the iterator is torn down, and
that teardown **blocks until curl finishes downloading the whole tar**. The validation build
opened a shard, took 40 images, and abandoned it: cleanup then sat waiting on the remaining
~4 GB. Twelve threads doing that in parallel, 134 times, is not a build, it is a slow strangle.
webdataset is gone. Module 6 reads tar bytes directly (~90 lines, unit-tested against byte
offsets) and owns its subprocess: explicit `close()` that kills curl, `try/finally` around the
sampler loop so a worker shutdown cannot leak, and a bounded end byte on every request so curl
exits by itself. The diversity cell now runs a `pgrep` census before and after dropping a
dataloader and **asserts** the streams are gone.

## 3. The bandwidth trap that made the run pointless anyway

v5.2 advanced into a shard by RECORD offset with `islice`, which downloads and discards every
record before the window. Cycle 13 meant ~2.4 GB of throwaway traffic to reach 74 MB of usable
images: a ~40:1 waste ratio, and on Kaggle's network that, not the GPUs, sets the step rate.
The full 30k-step run would have pulled multiple TB to train on ~90 GB.

Windows are now **HTTP Range requests** on real shard sizes read from the repo tree, so nothing
is downloaded except what is trained on. Tar cannot seek, so a range starting mid-archive is
resynchronised by scanning for a 512-byte block whose `ustar` magic *and* checksum both pass
(a false positive inside JPEG payload is effectively impossible). Verified on synthetic shards:
arbitrary offsets recover clean JPEGs, consecutive windows share zero records. Preflight now
asserts the CDN returns `206` with an exact-length body, because if Range were silently ignored
every window would collapse to byte 0 and we would be back to v5's prefix trap.

## 4. The family map was broken, and its own output said so

`real   47 exact classes` -- with 46 real shards. The old `[A-Za-z]+` prefix rule put a fake
method whose name starts with `Real` into the **real** family, so a generator scored as genuine
at family level. `e   2 exact classes` was `E4S` and `e4e` merged into a family called "e". And
79 of 88 fake methods were singletons, which makes `family_top1` arithmetically identical to
`fine_top1` and the user-facing roll-up decoration. Now: an explicit lineage table matched on
substrings, `"real"` reserved for genuine sources and asserted, full method stem as fallback
(never a one-letter prefix). The cell prints which families are real groupings and which are
singletons, so you can extend the table from your own output.

## 5. Restartability, since sessions do die

The validation build caches **per shard** under `/kaggle/working`, so the 100 shards the last
session collected would not have been lost: a rerun resumes and finishes the remaining 34.
Training already resumes from the newest checkpoint, and the 9.5 h wall-clock stop still
guarantees eval and export actually run.

## Still true from v5.2

Autocast inside `forward` (DataParallel runs replicas in their own threads, so Trainer's outer
autocast never reaches them), `channels_last`, `IMAGES_PER_STEP = bs x N_GPU x accum` asserted
against Trainer, no torch/torchvision reinstall, throughput logging, preflight.

`t4x2` preset: 32/GPU x 2 GPUs x accum 1 = **64 images/step**, 30,000 steps, eval every 1,500.
Watch the `[rate]` line: two T4s in fp16 top out near 170-190 img/s on B4 at 256px. Below ~120
the ceiling is 4 CPU cores doing JPEG decode plus albumentations, or the network. Raise
`CFG["workers"]` or thin the augmentation. Lowering batch size will not help.


# DeepTrace v5.1 - EfficientNet generator attribution model

Trains a single model that answers two questions about an image:

1. **What made it?** (134-way generator attribution, primary objective, weight 1.0)
2. **Is it real or fake?** (binary verdict, auxiliary, weight 0.6)

Built on [ScaleDF](https://huggingface.co/datasets/WenhaoWang/ScaleDF): one class per `.tar`
shard, streamed with WebDataset. `ScaleDF/train` holds 46 real-source shards and 88 fake-method
shards (~20.7M images total, ~155k per shard, stored as 256x256 JPEG).

---

## Changes from v5 (these were bugs, not preferences)

**1. The prefix trap (the big one).** v5 set `SLOT_LIFETIME = 256` and every slot rotation
reopened a shard **from byte zero**. Tar streams cannot seek, so training only ever saw the first
~320 records of each shard: ~43k images, **0.21% of ScaleDF**, each replayed ~17 times across a
15k-step run. That is not streaming 20M images, that is memorising 43k. Fixed with a **shard deck**:
each group's shards are visited in shuffled order, once per cycle, and every cycle reads the *next*
contiguous window (`floor + cycle*stride`). Worker ids are folded into the stride so parallel
workers never read the same images. Result: ~1 pass over ~700k unique images, no repeats, no
bandwidth spent skipping, and every class gets an equal number of turns.

**2. Train/val leakage.** v5's validation set took the *first* 40/21 records of each training
shard, i.e. exactly the window training was locked onto. Every validation image was a training
image, so every v5 metric was meaningless. Fixed by reserving records `[0, VAL_RESERVE)` for
validation and starting every training stream at `VAL_RESERVE` or later. Enforced by assertion.

**3. Class weights were inverted.** Group-balanced sampling (50% real / 50% fake) makes each real
class **1.91x more frequent** than each fake class. v5's "tempered inverse frequency" then gave
real classes **1.38x more** weight, compounding to ~2.6x. That is the exact v3 bias the design
doc claims to have removed. Correct tempered inverse frequency (power 0.5) is real **0.80** /
fake **1.11**, ratio 0.72.

**4. Batch diversity depended on worker count.** Diversity comes from the *per-worker* slot pool,
because each batch is produced by one worker. v5 divided a 32-slot pool across workers, so the
`num_workers=4` preset trained on ~8-class batches while the `num_workers=0` sanity check
measured 16 and reported success. Pool size is now per worker.

**5. Resolution.** ScaleDF ships 256x256 images. v5 read `processor.size` (467 for B4) and
upsampled everything, smearing the high-frequency generator fingerprints that are the entire
signal, at ~3x the compute. Now trained at native 256 with pixel-preserving crops.

**6. EMA was dead code.** Shadow weights were updated every step and never used for evaluation,
selection, or export: pure overhead. Now swapped in for eval and checkpointing, out before the
next step.

**7. `json` was never imported**, so the export cell raised `NameError` after a multi-hour run.

**8. Auth.** Query-string tokens are not honoured on HF resolve URLs. Shards now stream through
`pipe:curl` with an `Authorization` header, token read from the environment, never printed.

**9. The official held-out split was silently discarded.** `ScaleDF/val` holds *different*
domains and methods, so v5's name-match filter dropped nearly all of it. Those shards cannot
score 134-way attribution, but they are the only honest measure of cross-generator
generalisation, so they are now evaluated as a binary benchmark.


## 1. Environment

In [ ]:
# Module 1: Environment setup -- Kaggle-safe
#
# Kaggle's image already ships a torch/torchvision built against the exact driver on the T4
# nodes. Reinstalling either one (or letting a dependency drag them in) swaps in a generic
# wheel, the CUDA build stops matching, and you get a cryptic device-side assert or a
# "no kernel image is available" 40 minutes into a run. So: pin what we need, never touch
# torch/torchvision/numpy, and pass --no-warn-conflicts so pip's noise does not look like an error.
import subprocess, sys, importlib

# webdataset is gone on purpose. Its pipe: reader owns a curl subprocess it will not let go
# of: abandon a partially-read shard and teardown blocks until curl has finished downloading
# the WHOLE multi-GB tar. Twelve of those at once is what killed the last session at shard
# 100/134. Module 6 now reads tar bytes directly and kills its own curl (~90 lines, tested).
PKGS = [
    "transformers>=4.46,<5",     # Trainer(processing_class=...)
    "accelerate>=0.26",          # Trainer hard requirement
    "albumentations==2.0.8",     # last MIT release before the AGPL fork
    "albucore==0.0.24",          # albumentations 2.0.8 pins this exact build
    "evaluate",
    "huggingface_hub>=0.26",
]

# Kaggle's preinstalled opencv already works headless. Installing opencv-python-headless on
# top of it leaves two cv2 distributions fighting over the same import name.
try:
    importlib.import_module("cv2")
    print("cv2 already present, skipping opencv install.")
except Exception:
    PKGS.append("opencv-python-headless")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-warn-conflicts", *PKGS],
               check=True)

import torch, transformers
print(f"torch {torch.__version__} (cuda {torch.version.cuda}) | "
      f"transformers {transformers.__version__} | GPUs visible: {torch.cuda.device_count()}")
assert torch.cuda.is_available(), (
    "No CUDA device. In Kaggle: Settings -> Accelerator -> GPU T4 x2."
)


## 2. Reproducibility, device, authentication

In [ ]:
# Module 2: Reproducibility, GPU topology, AMP policy, HF authentication
import os

# ---- Multi-GPU policy (must be set before anything initialises CUDA) --------------------
# ---- FIX 10: THE CRASH THIS TIME ------------------------------------------------------
#   AcceleratorError: Caught AcceleratorError in replica 0 on device 0
#   ... modeling_efficientnet.py line 181, self.depthwise_conv(hidden_states)
#   torch.AcceleratorError: CUDA error: an illegal memory access was encountered
#
# Kaggle's "GPU T4 x2" gives two 15 GB Turing cards in ONE process, so HF Trainer silently
# wraps the model in nn.DataParallel. DataParallel replicates the module by *coalescing and
# flattening* the parameters (comm.broadcast_coalesced -> flatten_dense_tensors), and a
# flattened buffer has no memory format: the NHWC (channels_last) conv weights come out the
# other side carrying NCHW strides over NHWC bytes. Replica 0 keeps the real channels_last
# params, the other replica does not, and cudnn then picks an NHWC depthwise kernel for a
# tensor whose strides lie about its layout. On EfficientNet-B4 that is 32 grouped depthwise
# convs per forward, in fp16, launched from two Python threads at once. It faulted mid-eval
# rather than on the probe because the fault is asynchronous: the probe and the cudnn warmup
# only ever ran one shape at a time and returned before the bad kernel landed.
#
# Two ways out. Drop channels_last (keeps both GPUs, loses the 15-25% NHWC win on depthwise
# convs), or drop DataParallel (keeps NHWC, loses the second card). This notebook does the
# second, because the run's own preflight already says the ceiling is elsewhere:
#   "CPU cores available: 4 -- this, not the GPUs, is usually the throughput ceiling"
# The v5.3 plan was 1.92 M images in 9.5 h = ~56 img/s, and a single T4 does ~90-120 img/s on
# B4 at 256px in fp16. The pipeline is decode- and network-bound, so the second T4 was mostly
# waiting anyway, while costing a replicate + gather every single step -- and, per the above,
# silently computing half of every batch with mis-strided weights even when it did not crash.
#
# Set this back to True only if you also set USE_CHANNELS_LAST = False in Module 8. The two
# are mutually exclusive; Module 8 now enforces that automatically.
USE_BOTH_GPUS = False
if not USE_BOTH_GPUS:
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Fragmentation guard: 40 shard streams + DataParallel replication churns the allocator.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import io, re, json, math, random, itertools
import numpy as np
import torch
from PIL import Image

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

N_GPU = 0
GPU_NAMES = []
if torch.cuda.is_available():
    device = torch.device("cuda")
    torch.cuda.manual_seed_all(SEED)
    N_GPU = torch.cuda.device_count()
    GPU_NAMES = [torch.cuda.get_device_name(i) for i in range(N_GPU)]
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    TOTAL_VRAM = sum(torch.cuda.get_device_properties(i).total_memory for i in range(N_GPU)) / 1e9
    _cc = torch.cuda.get_device_capability(0)
    for i, n in enumerate(GPU_NAMES):
        print(f"GPU {i}: {n} ({torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB)")
    print(f"Compute capability sm_{_cc[0]}{_cc[1]} | total VRAM {TOTAL_VRAM:.1f} GB")
    # TF32 is Ampere+ only. Harmless no-op on Turing, kept so the notebook is portable.
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    IS_MULTI_GPU = N_GPU > 1
    IS_TURING = _cc[0] == 7
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps"); gpu_mem = TOTAL_VRAM = 0.0
    IS_MULTI_GPU = IS_TURING = False
    print("Using Apple Silicon MPS")
else:
    device = torch.device("cpu"); gpu_mem = TOTAL_VRAM = 0.0
    IS_MULTI_GPU = IS_TURING = False
    print("WARNING: No GPU detected. Training will be impractically slow.")

# ---- AMP policy ------------------------------------------------------------------------
# THE TRAP, and the last run walked straight into it: on torch 2.10 + CUDA 12.8,
# torch.cuda.is_bf16_supported() returns True on a Tesla T4. Its default argument is
# including_emulation=True, and Turing can *emulate* bf16 -- in software, with no tensor-core
# path. The previous run therefore printed "AMP: bf16 (Turing has no bf16 tensor cores)",
# a sentence that contradicts itself, set bf16=True in TrainingArguments, and would have
# trained every conv through the emulation path: slower than fp32, never mind fp16.
#
# bf16 tensor cores start at Ampere (sm_80). Gate on compute capability, not on a query whose
# answer means "will run" rather than "will run fast".
_ccs = [torch.cuda.get_device_capability(i) for i in range(N_GPU)] if N_GPU else []
BF16_TENSOR_CORES = bool(_ccs) and all(c[0] >= 8 for c in _ccs)
try:
    _torch_claims_bf16 = torch.cuda.is_bf16_supported(including_emulation=False)
except TypeError:                                   # older torch has no such argument
    _torch_claims_bf16 = torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False

USE_BF16 = bool(BF16_TENSOR_CORES and _torch_claims_bf16)
USE_AMP = bool(torch.cuda.is_available())
AMP_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
if torch.cuda.is_available():
    print(f"AMP: {'bf16' if USE_BF16 else 'fp16 + GradScaler'} "
          f"(sm_{_ccs[0][0]}{_ccs[0][1]}, bf16 tensor cores: {BF16_TENSOR_CORES})")
    if not BF16_TENSOR_CORES:
        try:
            _emul = torch.cuda.is_bf16_supported()
        except Exception:
            _emul = None
        print(f"  torch.cuda.is_bf16_supported() says {_emul} here -- that includes software "
              f"emulation. Ignored: fp16 is the only fast path on this card.")

# ---- Working directory ------------------------------------------------------------------
# /kaggle/working persists into the notebook output (and survives a session restart), so the
# validation cache is built once instead of once per session. ~20 GB budget, we use <1 GB.
WORK_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
print(f"Work dir: {WORK_DIR}")

# ---- HuggingFace auth: Kaggle Secrets -> Colab Secrets -> interactive prompt -------------
try:
    from kaggle_secrets import UserSecretsClient
    _hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF token loaded from Kaggle Secrets.")
except Exception:
    try:
        from google.colab import userdata
        _hf_token = userdata.get("HF_TOKEN")
        print("HF token loaded from Colab Secrets.")
    except Exception:
        import getpass
        _hf_token = getpass.getpass("Enter your HuggingFace token: ")

assert _hf_token, (
    "HF_TOKEN is required to stream ScaleDF. In Kaggle: Add-ons -> Secrets -> "
    "add HF_TOKEN and tick it for this notebook."
)
# Exported so the curl subprocesses webdataset spawns can authenticate via header.
# HF ignores ?token= on resolve URLs, which is why v5's URL suffix did nothing.
os.environ["HF_TOKEN"] = _hf_token

try:
    from huggingface_hub import login
    login(token=_hf_token, add_to_git_credential=False)
    print("Authenticated with HuggingFace Hub.")
except Exception as e:
    print(f"login() warning ({e}); using env var fallback.")

assert os.system("curl --version > /dev/null 2>&1") == 0, "curl is required for shard streaming."
print("curl available.")


### Preflight checks

In [ ]:
# Module 2b: Preflight -- fail in 20 seconds instead of 20 minutes
#
# Every check here maps to a real Kaggle failure that otherwise surfaces as a confusing
# traceback deep inside a dataloader worker, long after you have stopped watching.
import shutil, urllib.request

def _check(label, fn, hint):
    try:
        detail = fn()
        print(f"  [ok]   {label}{f': {detail}' if detail else ''}")
        return True
    except Exception as e:
        print(f"  [FAIL] {label}: {e}\n         -> {hint}")
        return False

print("Preflight:")
ok = True

ok &= _check(
    "Internet reachable",
    lambda: urllib.request.urlopen("https://huggingface.co", timeout=15).status and "huggingface.co",
    "Kaggle: Notebook Settings -> Internet -> ON. Required, ScaleDF is streamed, not downloaded.",
)

def _repo():
    from huggingface_hub import HfApi
    info = HfApi().dataset_info("WenhaoWang/ScaleDF")
    return f"{info.id} accessible"
ok &= _check(
    "ScaleDF access with your token", _repo,
    "Open huggingface.co/datasets/WenhaoWang/ScaleDF, accept the terms, and make sure the "
    "token has read access to gated repos.",
)

def _curlauth():
    import subprocess
    r = subprocess.run(
        'curl -s -o /dev/null -w "%{http_code}" -L -H "Authorization: Bearer $HF_TOKEN" '
        '"https://huggingface.co/api/datasets/WenhaoWang/ScaleDF"',
        shell=True, capture_output=True, text=True, timeout=60)
    assert r.stdout.strip() == "200", f"HTTP {r.stdout.strip()} from curl subprocess"
    return "curl inherits HF_TOKEN (200)"
ok &= _check(
    "Shard streaming auth", _curlauth,
    "webdataset streams through curl in a subprocess. If curl cannot see HF_TOKEN, every "
    "shard silently decodes to zero samples.",
)

def _range():
    import subprocess
    r = subprocess.run(
        'curl -s -w "|%{http_code}|%{size_download}" -o /dev/null -L -r 0-1023 '
        '-H "Authorization: Bearer $HF_TOKEN" '
        '"https://huggingface.co/datasets/WenhaoWang/ScaleDF/resolve/main/'
        'ScaleDF/train/000000AFAD.tar"',
        shell=True, capture_output=True, text=True, timeout=90)
    _, codeu, size = r.stdout.rsplit("|", 2)
    assert codeu == "206", f"expected HTTP 206 Partial Content, got {codeu}"
    assert int(float(size)) == 1024, f"asked for 1024 bytes, server sent {size}"
    return "HTTP 206, exact 1 KB window"
ok &= _check(
    "Byte-range requests honoured", _range,
    "Module 6 seeks into shards with Range requests instead of downloading and discarding "
    "gigabytes. If the CDN ignores Range it returns 200 and the whole file, and every window "
    "offset silently becomes byte 0 -- the v5 prefix trap all over again.",
)

ok &= _check(
    "Disk headroom in work dir",
    lambda: f"{shutil.disk_usage(WORK_DIR).free / 1e9:.1f} GB free",
    "Checkpoints + val cache need ~2 GB. Clear /kaggle/working.",
)

def _gpus():
    assert N_GPU >= 1, "no CUDA device"
    free, total = torch.cuda.mem_get_info(0)
    assert free / 1e9 > 12, f"only {free / 1e9:.1f} GB free on GPU 0, something else is resident"
    return f"{N_GPU} x {GPU_NAMES[0]}, {free / 1e9:.1f} GB free"
ok &= _check("GPUs idle and available", _gpus,
             "Restart the kernel; a previous run is still holding VRAM.")

print(f"\nCPU cores available: {os.cpu_count()}  (Kaggle T4 x2 gives 4 -- this, not the GPUs, "
      f"is usually the throughput ceiling)")
assert ok, "Preflight failed. Fix the [FAIL] lines above before running anything else."
print("Preflight passed.")


## 3. Resolution and normalization

In [ ]:
# Module 3: Resolution and normalization statistics
from transformers import AutoImageProcessor

MODEL_ID = "google/efficientnet-b4"
processor = AutoImageProcessor.from_pretrained(MODEL_ID)

# ScaleDF ships 256x256 JPEGs. Do NOT use the processor default (467px for B4): upsampling
# invents high-frequency content and destroys the resampling fingerprints that separate
# generators, and costs ~3x the compute to do it. Train at the data's native resolution.
NATIVE_SIZE = 256
IMAGE_SIZE = NATIVE_SIZE

_ps = processor.size
_proc_px = _ps.get("height", _ps.get("shortest_edge", "?"))
NORM_MEAN = processor.image_mean
NORM_STD = processor.image_std

# Keep the exported preprocessor_config.json consistent with what we actually feed the model.
processor.size = {"height": IMAGE_SIZE, "width": IMAGE_SIZE}
if getattr(processor, "crop_size", None):
    processor.crop_size = {"height": IMAGE_SIZE, "width": IMAGE_SIZE}

print(f"Processor default: {_proc_px}px (ignored)")
print(f"Training at:       {IMAGE_SIZE}x{IMAGE_SIZE} (ScaleDF native)")
print(f"Normalize mean:    {NORM_MEAN}")
print(f"Normalize std:     {NORM_STD}")


## 4. Label discovery: classes, families, shard URLs

In [ ]:
# Module 4: Discover classes, build shard URLs, map generator families
from huggingface_hub import HfApi
from collections import Counter

api = HfApi()
REPO_ID = "WenhaoWang/ScaleDF"
BASE_URL = f"https://huggingface.co/datasets/{REPO_ID}/resolve/main/"

# Shards are addressed by plain resolve URL. Module 6 builds its own curl argv (and its own
# 0600 config file for the auth header) so it can own, bound, and kill each subprocess.

print("Discovering training shards from ScaleDF/train ...")
_tree = list(api.list_repo_tree(REPO_ID, path_in_repo="ScaleDF/train", repo_type="dataset"))
train_tars = sorted([item.path for item in _tree if item.path.endswith(".tar")])
assert train_tars, "No training shards found. Check your HF token and network."

# Shard byte sizes, straight from the repo tree. Module 6 lays out per-cycle read windows in
# BYTES and seeks with Range requests, so it needs to know how big each shard actually is.
SHARD_BYTES = {i.path: int(i.size) for i in _tree
               if i.path.endswith(".tar") and getattr(i, "size", None)}
_median = int(np.median(list(SHARD_BYTES.values()))) if SHARD_BYTES else 4 << 30
for t in train_tars:
    SHARD_BYTES.setdefault(t, _median)
print(f"Shard sizes: min {min(SHARD_BYTES[t] for t in train_tars) / 2**30:.2f} GB | "
      f"median {_median / 2**30:.2f} GB | "
      f"max {max(SHARD_BYTES[t] for t in train_tars) / 2**30:.2f} GB | "
      f"total {sum(SHARD_BYTES[t] for t in train_tars) / 2**40:.2f} TB")

class_names = [os.path.basename(t)[:-4] for t in train_tars]
NUM_CLASSES = len(class_names)
label2id = {n: i for i, n in enumerate(class_names)}
id2label = {i: n for i, n in enumerate(class_names)}

# ScaleDF convention: shards prefixed 000000 are real sources, all others are fake methods.
def is_real_name(name):
    return name.startswith("000000")

real_class_ids = [i for i, n in enumerate(class_names) if is_real_name(n)]
fake_class_ids = [i for i, n in enumerate(class_names) if not is_real_name(n)]
REAL_ID_SET, FAKE_ID_SET = set(real_class_ids), set(fake_class_ids)
assert real_class_ids and fake_class_ids, "Need both real and fake shards."

# (url, label, name) triples. Labels travel with the URL so nothing has to be recovered later
# by string-parsing a curl command.
# (url, label, name, size_bytes). The URL is now the plain resolve URL, not a curl command:
# Module 6 builds its own argv so it can own and kill the subprocess, and so the token lives in
# a 0600 config file instead of a command line visible in `ps`.
train_shards = [(BASE_URL + t, label2id[os.path.basename(t)[:-4]],
                 os.path.basename(t)[:-4], SHARD_BYTES[t]) for t in train_tars]

# --- Generator families, rebuilt ---------------------------------------------------------
# The leading-alpha-prefix scheme produced 80 families for 134 classes and two hard failures,
# both visible in the last run's output:
#
#   "real   47 exact classes"  <- with only 46 real shards. A fake method whose name starts
#       with "Real" (prefix regex stops at the first non-letter) landed in the "real" family,
#       so that generator scored as GENUINE at family level and quietly polluted every
#       family metric and the user-facing roll-up.
#   "e      2 exact classes"   <- E4S and e4e collapsed into a family called "e", while
#       distinct lineages stayed separate. 79 of 88 fake methods ended up as singletons, which
#       makes family_top1 arithmetically identical to fine_top1 and the roll-up
#       ("likely StableDiffusion family") pure decoration.
#
# Now: an explicit lineage table matched on substrings, "real" reserved for genuine sources by
# construction, and the full method stem as the fallback (never a one-letter prefix). Print the
# table below, then extend FAMILY_LINEAGES for whatever your run reports as a singleton.
FAMILY_LINEAGES = [
    ("stable-diffusion", ("stablediffusion", "stable_diffusion", "sdxl", "sd1", "sd2", "sd3",
                          "sdv", "sd_", "dreamshaper", "realvis", "juggernaut", "epicrealism",
                          "majicmix", "chilloutmix", "deliberate", "playground", "ldm")),
    ("flux",             ("flux",)),
    ("dalle",            ("dalle", "dall_e", "dall-e")),
    ("midjourney",       ("midjourney",)),
    ("cogview",          ("cogview",)),
    ("pixart",           ("pixart",)),
    ("kandinsky",        ("kandinsky",)),
    ("hunyuan",          ("hunyuan",)),
    ("video-diffusion",  ("cogvideo", "svd", "opensora", "mochi", "kling", "sora", "pika",
                          "runway", "wan", "seedance", "veo", "latte")),
    ("gan",              ("stylegan", "progan", "pggan", "biggan", "stargan", "attgan",
                          "vqgan", "gan_", "_gan", "styleswin")),
    ("gan-inversion",    ("e4e", "e4s", "psp", "restyle", "hyperstyle", "hfgi")),
    ("ffpp",             ("ffpp", "faceforensics", "deepfakes", "face2face", "faceswap",
                          "neuraltextures", "faceshifter", "simswap", "inswapper", "blendface",
                          "uniface", "mobileswap", "fsgan", "3dswap")),
    ("portrait-anim",    ("aniportrait", "liveportrait", "sadtalker", "echomimic", "hallo",
                          "animatediff", "wav2lip", "memo", "vexpress", "joyvasa", "dreamtalk",
                          "makeittalk", "styleheat", "talklip", "facevid2vid", "mcnet", "tpsm",
                          "fomm", "sparsectrl", "magicanimate")),
    ("attribute-edit",   ("stylecl", "latentdiff", "e4t", "diffae", "starganv", "styleres",
                          "cyclegan", "ganimation", "interfacegan", "talkface")),
    ("inpaint-edit",     ("inpaint", "instructpix2pix", "controlnet", "ip_adapter", "ipadapter",
                          "repaint", "sdedit", "blended")),
    ("restoration",      ("codeformer", "gfpgan", "restoreformer", "esrgan", "swinir")),
]
_LINEAGE_HITS = Counter()

def generator_family(name):
    """Coarse lineage for user-facing output. 'real' is reserved for genuine sources."""
    if is_real_name(name):
        return "real"
    stem = re.sub(r"_faces$", "", name).lower()
    for fam, keys in FAMILY_LINEAGES:
        if any(k in stem for k in keys):
            _LINEAGE_HITS[fam] += 1
            return fam
    return stem or name.lower()        # its own family; never a truncated prefix, never "real"

CLASS_TO_FAMILY = {i: generator_family(n) for i, n in id2label.items()}
FAMILY_NAMES = sorted(set(CLASS_TO_FAMILY.values()))

# Guard rails for exactly the two bugs above.
_real_fam = [i for i in range(NUM_CLASSES) if CLASS_TO_FAMILY[i] == "real"]
assert set(_real_fam) == set(real_class_ids), (
    f"'real' family holds {len(_real_fam)} classes but there are {len(real_class_ids)} real "
    f"shards. Offenders: {[id2label[i] for i in set(_real_fam) - set(real_class_ids)]}")
assert all(CLASS_TO_FAMILY[i] != "real" for i in fake_class_ids)
family2id = {f: i for i, f in enumerate(FAMILY_NAMES)}
CLASS_TO_FAMILY_ID = np.array([family2id[CLASS_TO_FAMILY[i]] for i in range(NUM_CLASSES)])
REAL_FAMILY_ID = family2id["real"]

print(f"\nTotal classes:      {NUM_CLASSES}")
print(f"Real sources:       {len(real_class_ids)}")
print(f"Fake methods:       {len(fake_class_ids)}")
print(f"Generator families: {len(FAMILY_NAMES)}")
_fam_counts = Counter(CLASS_TO_FAMILY[i] for i in fake_class_ids)
_grouped = {f: c for f, c in _fam_counts.items() if c > 1}
_singletons = sorted(f for f, c in _fam_counts.items() if c == 1)
print("\nFake families with more than one method (these are what the roll-up can actually "
      "generalise over):")
for family, count in sorted(_grouped.items(), key=lambda kv: -kv[1]):
    members = [id2label[i] for i in fake_class_ids if CLASS_TO_FAMILY[i] == family]
    print(f"  {family:<18} {count:>3}  {', '.join(sorted(members)[:6])}"
          f"{' ...' if count > 6 else ''}")
print(f"\nGrouped: {sum(_grouped.values())}/{len(fake_class_ids)} fake methods in "
      f"{len(_grouped)} families | singletons: {len(_singletons)}")
if _singletons:
    print("Singleton families (family_top1 == fine_top1 for these; add them to "
          "FAMILY_LINEAGES if any share a lineage):")
    for i in range(0, len(_singletons), 6):
        print("   " + ", ".join(_singletons[i:i + 6]))
assert CLASS_TO_FAMILY[real_class_ids[0]] == "real"

print(f"\nUniform shard prior (fake): {len(fake_class_ids) / NUM_CLASSES:.3f}")
print("  This was v2's frozen binary precision. The group-balanced sampler removes it.")


In [ ]:
# Module 4b: The official held-out split (cross-generator generalisation benchmark)
#
# ScaleDF/val contains DIFFERENT real domains and DIFFERENT fake methods from train. v5 filtered
# these shards against label2id and silently dropped almost all of them. They cannot score
# 134-way attribution (those classes are not in our label space), but they are the only honest
# measure of what matters in production: does this hold up on a generator it has never seen?
# Kept as a BINARY benchmark.
heldout_shards = []       # (url, binary_label, name, size_bytes), 1 = fake
heldout_seen_names = []
HELDOUT_SPLIT = None

for _candidate in ("ScaleDF/val", "ScaleDF/validation", "ScaleDF/test"):
    try:
        _t = api.list_repo_tree(REPO_ID, path_in_repo=_candidate, repo_type="dataset")
        _t = list(_t)
        _tars = sorted([i.path for i in _t if i.path.endswith(".tar")])
    except Exception:
        continue
    if not _tars:
        continue
    _i_size = {i.path: i for i in _t if i.path.endswith(".tar")}
    HELDOUT_SPLIT = _candidate
    for t in _tars:
        stem = os.path.basename(t)[:-4]
        heldout_shards.append((BASE_URL + t, 0 if is_real_name(stem) else 1, stem,
                               int(getattr(_i_size.get(t), "size", 0) or (256 << 20))))
        if stem in label2id:
            heldout_seen_names.append(stem)
    break

if HELDOUT_SPLIT:
    _nr = sum(1 for _s in heldout_shards if _s[1] == 0)
    print(f"Held-out split '{HELDOUT_SPLIT}': {len(heldout_shards)} shards "
          f"({_nr} real / {len(heldout_shards) - _nr} fake)")
    print(f"  {len(heldout_seen_names)} share a class with train, "
          f"{len(heldout_shards) - len(heldout_seen_names)} are UNSEEN domains/methods")
    print("  -> scored as a binary generalisation benchmark, not 134-way attribution")
else:
    print("No held-out split found. Cross-generator generalisation cannot be measured.")


## 5. Two-track forensic augmentation

In [ ]:
# Module 5: Two-track forensic augmentation
#
# Track 1 (~55%): pixel-preserving crop only. Generator fingerprints live in the high
#   frequencies, so the clean track must not resample. v5 ran RandomResizedCrop with
#   scale=(0.65, 1.0) on EVERY sample, rescaling 100% of the data and smearing exactly the
#   signal we want. Replaced with a native-resolution crop.
# Track 2 (~45%): one social-media style degradation, so detection survives re-encoding.
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2

# ---- cv2 threading OFF, and it has to happen HERE, before any DataLoader forks -----------
# albumentations runs on OpenCV, and OpenCV keeps a persistent worker-thread pool. This cell
# runs transforms in the main process (the sample pull, the diversity probe, the preview
# grid), so that pool is live and holding its internal mutexes by the time DataLoader forks.
# fork() copies the locks but not the threads that would release them, so the first forked
# worker can block forever inside a transform: no traceback, no output, no CPU use. The first
# fork in this notebook is the eval dataloader, i.e. exactly the "Baseline (untrained)..."
# line the run stops on.
#
# Also the right call on throughput grounds: 3 workers x 4 OpenCV threads on a 4-core Kaggle
# box is 12 threads fighting for 4 cores. One thread per worker, no oversubscription.
cv2.setNumThreads(0)
cv2.ocl.setUseOpenCL(False)

geometry = [
    # Guarantees an IMAGE_SIZE output for any input without resampling native 256px sources.
    A.PadIfNeeded(min_height=IMAGE_SIZE, min_width=IMAGE_SIZE, border_mode=0, fill=0, p=1.0),
    A.RandomCrop(height=IMAGE_SIZE, width=IMAGE_SIZE, p=1.0),
    A.HorizontalFlip(p=0.5),
    # Light scale/rotate on a minority of samples, for robustness to resized uploads.
    A.Affine(translate_percent=(-0.04, 0.04), scale=(0.92, 1.08), rotate=(-8, 8), p=0.25),
    A.ColorJitter(brightness=(0.9, 1.1), contrast=(0.9, 1.1),
                  saturation=(0.9, 1.1), hue=(-0.03, 0.03), p=0.4),
]

degrade = A.OneOf([
    A.ImageCompression(quality_range=(40, 85), p=1.0),
    A.Downscale(scale_range=(0.5, 0.85), p=1.0),
    A.GaussianBlur(blur_limit=(3, 5), p=1.0),
    A.GaussNoise(std_range=(0.02, 0.07), p=1.0),
], p=0.45)

train_augmentations = A.Compose([
    *geometry,
    degrade,
    A.CoarseDropout(num_holes_range=(1, 2), hole_height_range=(0.05, 0.12),
                    hole_width_range=(0.05, 0.12), p=0.10),
    A.Normalize(mean=NORM_MEAN, std=NORM_STD),
    ToTensorV2(),
])

# Eval / inference: shortest-side fit then centre crop. A no-op on native 256px ScaleDF images
# (validation pixels stay untouched) and correct for arbitrary user uploads.
val_augmentations = A.Compose([
    A.SmallestMaxSize(max_size=IMAGE_SIZE, p=1.0),
    A.PadIfNeeded(min_height=IMAGE_SIZE, min_width=IMAGE_SIZE, border_mode=0, fill=0, p=1.0),
    A.CenterCrop(height=IMAGE_SIZE, width=IMAGE_SIZE, p=1.0),
    A.Normalize(mean=NORM_MEAN, std=NORM_STD),
    ToTensorV2(),
])

print(f"Augmentation ready at {IMAGE_SIZE}px: ~55% clean (no resampling), ~45% degraded")
print(f"  cv2 threads: {cv2.getNumThreads()} (0 = fork-safe, set before any dataloader worker)")


## 6. Streaming: class-mixed sampler and leak-free validation

In [ ]:
# Module 6: Byte-seeking tar reader, class-mixed sampler, leak-free validation
#
# Two things changed here, and both are the difference between "this run finishes" and "this
# run dies or crawls".
#
# 1. WE OWN THE SUBPROCESS. webdataset's pipe: reader spawns curl and only reaps it during
#    interpreter teardown of the iterator, which blocks until curl has finished the WHOLE tar.
#    Abandon a partially-read 4 GB shard and cleanup hangs on gigabytes you already decided you
#    did not want. Twelve of those in parallel is what stalled the validation build at shard
#    100/134 and took the session with it. Every stream below is a Popen we terminate on
#    purpose, and every request carries an explicit end byte so curl exits by itself anyway.
#
# 2. WE SEEK IN BYTES, NOT RECORDS. v5.1 "advanced" into a shard with islice, i.e. it
#    downloaded and threw away every record before the window. At cycle 13 that is ~2.4 GB
#    discarded to reach 74 MB of usable images: a ~40:1 waste ratio, which on Kaggle bandwidth
#    is the actual training bottleneck, not the GPUs. Windows are now HTTP Range requests, so
#    nothing is downloaded except what is trained on. Tar cannot seek, so a range that starts
#    mid-archive is resynchronised by scanning for a 512-byte header whose ustar magic AND
#    checksum both pass -- a false positive inside JPEG payload is effectively impossible.
import subprocess, pickle, stat, time
import concurrent.futures as cf

# --- Sampler geometry ---
PER_WORKER_POOL = 16    # concurrent shard streams PER DATALOADER WORKER.
                        # Batch diversity == per-worker pool, because each batch is produced by
                        # exactly one worker. v5 divided a global pool by worker count, so the
                        # 4-worker preset silently trained on ~8-class batches while the
                        # num_workers=0 sanity check measured 16 and reported success.
SLOT_LIFETIME = 1536    # images read per stream before rotating to the next shard in the deck.
REAL_SAMPLE_PROB = 0.5  # group-level balance, 50% real / 50% fake per draw
SLOT_SHUFFLE_BUF = 128  # decorrelates consecutive records. ScaleDF real sources include video
                        # datasets (300VW etc.) where neighbouring records are adjacent frames.
OPEN_STAGGER = 0.15     # FIX 4: seconds between opening slots at worker start. Without it all
                        # 3 workers open 16 streams each in the same instant, so HF sees 48
                        # simultaneous range requests from one IP and starts returning 429s and
                        # connection resets. Each one surfaces as a slot that yields nothing,
                        # gets recycled, and opens yet another connection: a feedback loop that
                        # reads as "training is mysteriously slow for the first few minutes".
                        # Costs ~2.4 s once per worker and removes the spike entirely.

# --- Byte layout ---
VAL_RESERVE_BYTES = 32 << 20   # bytes [0, 32 MB) of every shard belong to VALIDATION ONLY;
                               # every training window starts at or after this mark. This is
                               # what makes validation honest -- v5 drew val from the exact
                               # window it trained on, so every v5 number was measured on
                               # training data. Enforced by assertion in Module 6b.
VAL_HEAD_BYTES = 24 << 20      # how much of the reserve we actually fetch to fill the val set
RECORD_BYTES_EST = 48 << 10    # refined from real blob sizes once the val cache exists
VAL_BUILD_THREADS = 8          # 4 CPU cores, but this is pure network wait

VAL_PER_REAL_SHARD = 40
VAL_PER_FAKE_SHARD = 21
VAL_CACHE_PATH = os.path.join(WORK_DIR, "val_cache_v5_3.pkl")
VAL_STORE_SIZE = IMAGE_SIZE

def slot_window_bytes():
    """Bytes to request per slot lifetime, from the measured average record size."""
    return int(SLOT_LIFETIME * RECORD_BYTES_EST * 1.06) + (1 << 20)

# --- curl config: the token lives in a 0600 file, never in argv (argv is visible in `ps`) ----
CURL_CFG = os.path.join(WORK_DIR, ".hf_curl.cfg")
with open(CURL_CFG, "w") as _f:
    _f.write(
        'silent\nshow-error\nfail\nlocation\n'
        'retry = 6\nretry-delay = 2\nretry-all-errors\n'
        'connect-timeout = 20\n'
        # Throughput floor: without it a throttled connection stops delivering bytes and curl
        # waits forever, blocking a dataloader worker with no error and no traceback.
        'speed-limit = 1024\nspeed-time = 30\n'
        f'header = "Authorization: Bearer {os.environ["HF_TOKEN"]}"\n')
os.chmod(CURL_CFG, stat.S_IRUSR | stat.S_IWUSR)

BLOCK = 512
IMG_EXTS = ("jpg", "jpeg", "png", "webp", "bmp", "tiff", "tif", "ppm")

# Cheap skips: members we know are not stills. Keeps sniffing off multi-MB video payloads.
NON_IMG_EXTS = ("mp4", "avi", "mov", "mkv", "webm", "m4v", "wav", "mp3", "flac", "m4a",
                "json", "txt", "csv", "npy", "npz", "pt", "pkl", "yaml", "yml", "md")
SNIFF_MAX_BYTES = 8 << 20      # payloads up to this size get magic-byte sniffed when the
                               # filename carries no usable extension


def _looks_like_image(head):
    """Magic-byte test. Used when a member name has no (or an unknown) extension."""
    return (head[:3] == b"\xff\xd8\xff"                              # jpeg
            or head[:8] == b"\x89PNG\r\n\x1a\n"                      # png
            or (head[:4] == b"RIFF" and head[8:12] == b"WEBP")        # webp
            or head[:2] == b"BM"                                      # bmp
            or head[:4] in (b"II*\x00", b"MM\x00*")                   # tiff
            or head[:2] in (b"P5", b"P6"))                            # pgm/ppm


def _pax_path(payload):
    """Pull 'path=' out of a pax extended header block: '<len> key=value\n' records."""
    for field in (b"path=", b"linkpath="):
        i = payload.find(field)
        if i >= 0:
            j = payload.find(b"\n", i)
            raw = payload[i + len(field): j if j >= 0 else len(payload)]
            return raw.decode("utf-8", "replace")
    return None


def _parse_tar_header(blk):
    """Validate a 512-byte tar header -> (name, size, typeflag), else None."""
    if len(blk) < BLOCK or blk[257:262] != b"ustar":
        return None
    field = blk[148:156].split(b"\0")[0].strip()
    if not field:
        return None
    try:
        want = int(field, 8)
    except ValueError:
        return None
    probe = blk[:148] + b" " * 8 + blk[156:]
    if sum(probe) != want and sum(c - 256 if c > 127 else c for c in probe) != want:
        return None                                  # checksum: this is what makes resync safe
    try:
        name = blk[0:100].split(b"\0")[0].decode("utf-8", "replace")
        sz = blk[124:136].split(b"\0")[0].strip()
        size = int(sz, 8) if sz else 0
    except Exception:
        return None
    return name, size, blk[156:157]


class _Buffered:
    """Buffered reader over a raw pipe: exact reads, cheap skips, byte-wise header hunting."""

    def __init__(self, fp, chunk=1 << 20):
        self.fp, self.chunk, self.buf, self.eof = fp, chunk, b"", False

    def _fill(self, n):
        while len(self.buf) < n and not self.eof:
            try:
                d = self.fp.read(max(self.chunk, n - len(self.buf)))
            except Exception:
                d = b""
            if not d:
                self.eof = True
                break
            self.buf += d
        return len(self.buf) >= n

    def read(self, n):
        if not self._fill(n):
            out, self.buf = self.buf, b""
            return out
        out, self.buf = self.buf[:n], self.buf[n:]
        return out

    def skip(self, n):
        while n > 0:
            got = self.read(min(n, 1 << 20))
            if not got:
                return False
            n -= len(got)
        return True

    def resync(self, max_scan=64 << 20):
        """Advance to the first byte that begins a valid tar header (mid-archive range start)."""
        scanned = 0
        while True:
            if not self._fill(BLOCK * 4) and len(self.buf) < BLOCK:
                return False
            pos = 0
            while True:
                i = self.buf.find(b"ustar", pos)
                if i < 0:
                    break
                start = i - 257
                if start < 0:
                    pos = i + 1
                    continue
                if len(self.buf) < start + BLOCK:
                    break                            # need more bytes before testing this one
                if _parse_tar_header(self.buf[start:start + BLOCK]):
                    self.buf = self.buf[start:]
                    return True
                pos = i + 1
            keep = min(len(self.buf), BLOCK)          # retain a possibly split header
            scanned += len(self.buf) - keep
            self.buf = self.buf[len(self.buf) - keep:]
            if self.eof and len(self.buf) < BLOCK:
                return False
            if scanned > max_scan:
                return False


def iter_tar_images(fp, resync=False):
    """Yield encoded image payloads from a tar byte stream, tolerating range boundaries.

    Name resolution matters more than it looks. When a member path exceeds 100 bytes, tar does
    not store it in the header: GNU writes a './/././@LongLink' record (typeflag 'L') whose
    PAYLOAD holds the real path, and pax writes an extended header (typeflag 'x'). Either way
    the following member header carries a TRUNCATED name, usually with the extension chopped
    off. An extension-only test therefore rejects every record in such a shard, which is
    exactly what produced "yielded nothing: no images in reserved head" on Speaking_Faces,
    RAVDESS, YouTubeFaces and REFace_faces: 4 of 134 classes with zero validation samples and
    zero training images, silently.

    So: honour long-name records, and when the name still has no usable extension, decide on
    the payload's magic bytes instead of trusting the filename.
    """
    b = _Buffered(fp)
    if resync and not b.resync():
        return
    pending_name = None
    while True:
        blk = b.read(BLOCK)
        if len(blk) < BLOCK:
            return
        if blk == b"\0" * BLOCK:
            continue
        h = _parse_tar_header(blk)
        if h is None:
            return
        name, size, flag = h
        pad = (-size) % BLOCK

        # Extended-name records: read the payload, keep the path, move on to the real member.
        if flag in (b"L", b"K", b"x", b"g"):
            payload = b.read(size)
            if len(payload) < size:
                return
            b.skip(pad)
            if flag == b"L":
                pending_name = payload.split(b"\0")[0].decode("utf-8", "replace")
            elif flag == b"x":
                pending_name = _pax_path(payload) or pending_name
            continue

        if pending_name:
            name, pending_name = pending_name, None

        if flag not in (b"0", b"\0"):                 # dirs, links, sparse, vendor extensions
            if size and not b.skip(size + pad):
                return
            continue

        ext = os.path.splitext(name)[1].lower().lstrip(".")
        if ext in IMG_EXTS:
            data = b.read(size)
            if len(data) < size:
                return                               # truncated at the range boundary
            b.skip(pad)
            yield data
            continue

        # No usable extension: sniff, unless it is obviously not a still or too big to be one.
        if ext in NON_IMG_EXTS or size == 0 or size > SNIFF_MAX_BYTES:
            if size and not b.skip(size + pad):
                return
            continue
        data = b.read(size)
        if len(data) < size:
            return
        b.skip(pad)
        if _looks_like_image(data[:16]):
            yield data


class ShardWindow:
    """A curl subprocess streaming one byte range of one shard. Owns and kills its process."""

    def __init__(self, url, start=0, length=None):
        argv = ["curl", "-K", CURL_CFG]
        if start or length:
            end = "" if length is None else str(start + length - 1)
            argv += ["-r", f"{start}-{end}"]         # explicit end: curl finishes on its own
        argv.append(url)
        self.proc = subprocess.Popen(argv, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL,
                                     stdin=subprocess.DEVNULL, bufsize=1 << 20)
        self.gen = iter_tar_images(self.proc.stdout, resync=start > 0)
        self.closed = False

    def images(self, limit=None):
        n = 0
        for blob in self.gen:
            yield blob
            n += 1
            if limit is not None and n >= limit:
                return

    def close(self):
        """Kill curl now. Never wait for the remainder of a multi-GB shard."""
        if self.closed:
            return
        self.closed = True
        try:
            self.gen.close()
        except Exception:
            pass
        try:
            self.proc.stdout.close()
        except Exception:
            pass
        try:
            self.proc.terminate()
            self.proc.wait(timeout=5)
        except Exception:
            try:
                self.proc.kill()
                self.proc.wait(timeout=5)
            except Exception:
                pass

    def __del__(self):
        try:
            self.close()
        except Exception:
            pass


def to_rgb_array(img):
    """RGB uint8 array, rejecting broken or tiny images."""
    if img is None:
        return None
    try:
        arr = np.array(img.convert("RGB"))
        if arr.ndim != 3 or arr.shape[2] != 3 or min(arr.shape[:2]) < 20:
            return None
        return arr
    except Exception:
        return None


def decode_blob(blob):
    try:
        return to_rgb_array(Image.open(io.BytesIO(blob)))
    except Exception:
        return None


class ShardDeck:
    """Shuffled deck of shards handing out a fresh BYTE window each time it recycles.

    1. Class rotation: every shard in the group is visited exactly once per cycle, so no class
       goes thousands of steps without a gradient while another repeats.
    2. Coverage: cycle N reads bytes [floor + N*stride, +window), so each revisit walks forward
       into data nobody has read. Worker ids are folded into the stride, so parallel workers
       read disjoint windows instead of duplicating each other.
    """

    def __init__(self, shards, rng, floor, window, worker_id=0, num_workers=1):
        self.shards = list(shards)                   # (url, label, size_bytes)
        self.rng = rng
        self.floor = floor
        self.window = window
        self.stride = window * max(1, num_workers)
        self.offset0 = floor + window * worker_id
        self.deck = []
        self.cycle = -1
        self.wrapped = 0

    def draw(self):
        if not self.deck:
            self.deck = self.shards[:]
            self.rng.shuffle(self.deck)
            self.cycle += 1
        url, label, size = self.deck.pop()
        start = self.offset0 + self.cycle * self.stride
        span = max(1, size - self.floor - self.window)
        if start + self.window > size:               # wrap rather than run off the end
            start = self.floor + ((start - self.floor) % span)
            self.wrapped += 1
        return url, label, start


class ShardSlot:
    """One open shard window plus a small random-pop buffer."""

    def __init__(self, url, label, start, window, count, buf_size):
        self.win = ShardWindow(url, start, window)
        self.it = self.win.images(limit=count)
        self.label = label
        self.buf = []
        self.buf_size = buf_size
        self.exhausted = False

    def next(self, rng):
        while len(self.buf) < self.buf_size and not self.exhausted:
            try:
                self.buf.append(next(self.it))
            except StopIteration:
                self.exhausted = True
            except Exception:
                self.exhausted = True
        if not self.buf:
            raise StopIteration
        j = rng.randrange(len(self.buf))
        self.buf[j], self.buf[-1] = self.buf[-1], self.buf[j]
        return self.buf.pop()

    def close(self):
        self.buf = []
        self.win.close()


class ScaleDFTrainDataset(torch.utils.data.IterableDataset):
    """Infinite stream of class-mixed, real/fake-balanced training samples.

    Anti-collapse mechanism: PER_WORKER_POOL shard windows stay open concurrently and every
    draw picks a random slot, so a batch spans ~PER_WORKER_POOL classes instead of one.
    Single-class batches make BatchNorm statistics class-conditional and leave the gradient
    with no information about why an image is real or fake; the only stable solution under SGD
    is a constant prediction. No loss function fixes that.
    """

    def __init__(self, shards, augmentation, real_id_set,
                 pool_size=PER_WORKER_POOL, slot_lifetime=SLOT_LIFETIME,
                 real_prob=REAL_SAMPLE_PROB, start_floor=VAL_RESERVE_BYTES, seed=SEED):
        super().__init__()
        self.augmentation = augmentation
        self.pool_size = pool_size
        self.slot_lifetime = slot_lifetime
        self.real_prob = real_prob
        self.start_floor = start_floor
        self.seed = seed

        self.real = [(u, l, sz) for u, l, _, sz in shards if l in real_id_set]
        self.fake = [(u, l, sz) for u, l, _, sz in shards if l not in real_id_set]
        assert self.real and self.fake, "Need both real and fake shards."
        print(f"Sampler: {len(self.real)} real / {len(self.fake)} fake shards | "
              f"{pool_size} windows/worker | P(real)={real_prob} | "
              f"{slot_lifetime} imgs/window | byte floor {start_floor / 2**20:.0f} MB")

    def _open(self, deck):
        url, label, start = deck.draw()
        return ShardSlot(url, label, start, deck.window, self.slot_lifetime, SLOT_SHUFFLE_BUF)

    def __iter__(self):
        info = torch.utils.data.get_worker_info()
        wid = info.id if info is not None else 0
        nw = info.num_workers if info is not None else 1
        rng = random.Random((self.seed * 1000003) ^ (9973 * (wid + 1)))
        window = slot_window_bytes()

        decks = {
            "real": ShardDeck(self.real, rng, self.start_floor, window, wid, nw),
            "fake": ShardDeck(self.fake, rng, self.start_floor, window, wid, nw),
        }
        n_real = max(1, int(round(self.pool_size * self.real_prob)))
        n_fake = max(1, self.pool_size - n_real)
        slots = []
        for _grp in (["real"] * n_real) + (["fake"] * n_fake):
            slots.append(self._open(decks[_grp]))
            if OPEN_STAGGER:
                time.sleep(OPEN_STAGGER)     # see OPEN_STAGGER: never open 48 streams at once
        groups = ["real"] * n_real + ["fake"] * n_fake
        real_idx = list(range(n_real))
        fake_idx = list(range(n_real, n_real + n_fake))

        # try/finally is the whole point: a worker shutting down (early stop, an exception,
        # persistent_workers recycling) must not leave 16 curl processes streaming.
        try:
            while True:
                group = "real" if rng.random() < self.real_prob else "fake"
                i = rng.choice(real_idx if group == "real" else fake_idx)
                try:
                    blob = slots[i].next(rng)
                except StopIteration:
                    slots[i].close()                 # window done -> next shard in the deck
                    slots[i] = self._open(decks[groups[i]])
                    continue
                except Exception:
                    slots[i].close()                 # network hiccup -> next shard in the deck
                    slots[i] = self._open(decks[groups[i]])
                    continue

                arr = decode_blob(blob)
                if arr is None:
                    continue
                try:
                    pixel_values = self.augmentation(image=arr)["image"]
                except Exception:
                    continue
                yield {"pixel_values": pixel_values, "label": slots[i].label}
        finally:
            for s in slots:
                try:
                    s.close()
                except Exception:
                    pass


class ScaleDFEvalDataset(torch.utils.data.Dataset):
    """Fixed, group-balanced eval set held in RAM as encoded bytes.

    Built from the reserved head of each shard with one bounded Range request per shard, on a
    thread pool, and cached PER SHARD under WORK_DIR. Per-shard caching is not a nicety: the
    last session died at shard 100/134 and lost every byte of a 30-minute build. Now a restart
    resumes from what already landed.

    Original JPEG bytes are kept verbatim when the image already fits VAL_STORE_SIZE, so
    validation measures forensic fingerprints instead of our own re-encoding.
    """

    def __init__(self, shards, augmentation, real_id_set,
                 per_real_shard=VAL_PER_REAL_SHARD, per_fake_shard=VAL_PER_FAKE_SHARD,
                 head_bytes=VAL_HEAD_BYTES, cache_path=None, threads=VAL_BUILD_THREADS):
        super().__init__()
        self.augmentation = augmentation
        self.items = []
        assert max(per_real_shard, per_fake_shard) > 0

        parts_dir = (cache_path + ".parts") if cache_path else None
        if parts_dir:
            os.makedirs(parts_dir, exist_ok=True)

        def _part_path(name):
            return os.path.join(parts_dir, f"{name}.pkl") if parts_dir else None

        def _collect(job):
            n, (url, label, name, _size) = job
            cap = per_real_shard if (label in real_id_set) else per_fake_shard
            p = _part_path(name)
            if p and os.path.exists(p):
                try:
                    with open(p, "rb") as f:
                        return n, pickle.load(f), "cached"
                except Exception:
                    pass
            out = []
            # FIX 3: the retry used to widen to head_bytes * 4 (96 MB), reaching far past the
            # 32 MB validation reserve and straight into bytes the sampler trains on. That
            # silently reintroduced the train/val overlap this entire module exists to prevent,
            # for exactly the awkward shards that needed a retry. Retry wider, never outside
            # the reserve.
            for budget in (head_bytes, min(head_bytes * 4, VAL_RESERVE_BYTES)):
                out = []          # attempt 2 re-reads from byte 0; never append to attempt 1
                win = ShardWindow(url, 0, budget)
                try:
                    for blob in win.images():
                        if len(out) >= cap:
                            break
                        try:
                            img = Image.open(io.BytesIO(blob))
                            if min(img.size) < 20:
                                continue
                            if max(img.size) <= VAL_STORE_SIZE:
                                keep = blob                  # original pixels, byte for byte
                            else:
                                rgb = img.convert("RGB")
                                scale = VAL_STORE_SIZE / min(rgb.size)
                                if scale < 1.0:
                                    rgb = rgb.resize((max(1, round(rgb.size[0] * scale)),
                                                      max(1, round(rgb.size[1] * scale))),
                                                     Image.BICUBIC)
                                b = io.BytesIO()
                                rgb.save(b, format="JPEG", quality=97, subsampling=0)
                                keep = b.getvalue()
                            out.append((keep, label))
                        except Exception:
                            continue
                except Exception as e:
                    if not out:
                        return n, out, repr(e)
                finally:
                    win.close()
                if len(out) >= cap:
                    break                            # got the quota, no need to widen the range
            if p and out:
                try:
                    with open(p, "wb") as f:
                        pickle.dump(out, f, protocol=4)
                except Exception:
                    pass
            return n, out, None if out else "no images in reserved head"

        if cache_path and os.path.exists(cache_path):
            with open(cache_path, "rb") as f:
                self.items = pickle.load(f)
            print(f"Loaded {len(self.items)} eval samples from {os.path.basename(cache_path)}.")
            self._summarize(real_id_set)
            return

        cached_already = sum(1 for _, _, nm, _ in shards
                             if _part_path(nm) and os.path.exists(_part_path(nm)))
        print(f"Building eval set from the reserved head of {len(shards)} shards "
              f"({per_real_shard}/real, {per_fake_shard}/fake, {head_bytes / 2**20:.0f} MB "
              f"range each) on {threads} threads. {cached_already} already cached.")
        t0 = time.time()
        collected, done, failures = {}, 0, []
        with cf.ThreadPoolExecutor(max_workers=threads) as pool:
            futs = [pool.submit(_collect, (n, s)) for n, s in enumerate(shards)]
            for fut in cf.as_completed(futs):
                n, got, err = fut.result()
                collected[n] = got
                if err and err != "cached":
                    failures.append((shards[n][2], err))
                done += 1
                if done % 20 == 0 or done == len(shards):
                    tot = sum(len(v) for v in collected.values())
                    print(f"  {done}/{len(shards)} shards -> {tot} samples "
                          f"({time.time() - t0:.0f}s)")
        for n in range(len(shards)):
            self.items.extend(collected.get(n, []))
        for name, err in failures[:10]:
            print(f"  shard {name} yielded nothing: {err}")

        if cache_path:
            try:
                with open(cache_path, "wb") as f:
                    pickle.dump(self.items, f, protocol=4)
                print(f"Cached to {os.path.basename(cache_path)} "
                      f"({time.time() - t0:.0f}s total).")
            except Exception as e:
                print(f"Cache write failed: {e}")
        self._summarize(real_id_set)

    def _summarize(self, real_id_set):
        dist = Counter(l for _, l in self.items)
        n_real = sum(v for k, v in dist.items() if k in real_id_set)
        mb = sum(len(b) for b, _ in self.items) / 1e6
        print(f"Eval set: {len(self.items)} samples | {n_real} real / "
              f"{len(self.items) - n_real} fake | {len(dist)} distinct labels | {mb:.0f} MB")

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        blob, label = self.items[idx]
        arr = np.array(Image.open(io.BytesIO(blob)).convert("RGB"))
        return {"pixel_values": self.augmentation(image=arr)["image"], "label": label}


print("Dataset classes defined.\n")

print("Creating training dataset...")
train_ds = ScaleDFTrainDataset(train_shards, train_augmentations, REAL_ID_SET)

print("\nCreating balanced validation dataset (bytes disjoint from every training window)...")
val_ds = ScaleDFEvalDataset(train_shards, val_augmentations, REAL_ID_SET,
                            cache_path=VAL_CACHE_PATH)
assert len(val_ds) > 0, "Validation dataset is empty."

VAL_FAKE_PRIOR = float(np.mean([l in FAKE_ID_SET for _, l in val_ds.items]))
print(f"Val fake prior: {VAL_FAKE_PRIOR:.3f} (must be ~0.50 for honest metrics)")
assert 0.42 <= VAL_FAKE_PRIOR <= 0.58, "Validation set is not real/fake balanced."

# Calibrate the byte<->image conversion from real data instead of guessing. Every training
# window size, coverage number and bandwidth estimate downstream depends on this.
_mean_blob = float(np.mean([len(b) for b, _ in val_ds.items]))
RECORD_BYTES_EST = int(_mean_blob * 1.04) + BLOCK * 3      # + tar header, padding, sidecars
print(f"Measured record size: {_mean_blob / 1024:.1f} KB image -> "
      f"{RECORD_BYTES_EST / 1024:.1f} KB per record on the wire")
print(f"Slot window: {slot_window_bytes() / 2**20:.1f} MB for {SLOT_LIFETIME} images")


In [ ]:
# Module 6b: Leakage, coverage and rotation assertions
#
# Assertions, not comments. The byte reserve only means something if the sampler honours it,
# and "parallel workers read disjoint data" only means something if the offsets say so.
assert train_ds.start_floor >= VAL_RESERVE_BYTES, "Training windows may read validation bytes."
assert VAL_HEAD_BYTES <= VAL_RESERVE_BYTES, "Validation reads past its own reserve."

_W = slot_window_bytes()
_rng = random.Random(0)
_per_worker = {}
for _wid in range(3):
    _deck = ShardDeck(train_ds.fake, random.Random(0), VAL_RESERVE_BYTES, _W,
                      worker_id=_wid, num_workers=3)
    _draws = [_deck.draw() for _ in range(len(train_ds.fake) * 3)]   # three full deck cycles
    _starts = [s for _, _, s in _draws]
    assert min(_starts) >= VAL_RESERVE_BYTES, "A training window overlaps the val reserve."
    assert set(Counter(l for _, l, _ in _draws).values()) == {3}, \
        "Deck did not visit every shard equally per cycle."
    _per_worker[_wid] = set(_starts)
    if _wid == 0:
        _cycle_starts = sorted(set(_starts))

assert not (_per_worker[0] & _per_worker[1]), "Workers 0 and 1 read the same windows."
assert not (_per_worker[1] & _per_worker[2]), "Workers 1 and 2 read the same windows."

print(f"Leakage:  val owns bytes [0,{VAL_RESERVE_BYTES / 2**20:.0f} MB), training windows "
      f"start at {min(_per_worker[0]) / 2**20:.0f} MB+. Disjoint.")
print(f"Rotation: every fake shard drawn exactly 3x over 3 deck cycles "
      f"({len(train_ds.fake)} shards).")
print(f"Workers:  disjoint windows. worker0 cycle offsets "
      f"{[round(s / 2**20) for s in _cycle_starts]} MB, worker1 "
      f"{[round(s / 2**20) for s in sorted(_per_worker[1])]} MB")
print(f"Window:   {_W / 2**20:.1f} MB per slot lifetime, and every byte of it is trained on "
      f"(v5.1 discarded ~40x this much per window to reach the offset).")


## 7. Sanity checks: batch diversity

In [ ]:
# Module 7: Verify batch class diversity (the single most important check)
# If this fails, the sampler is degenerate and no amount of loss tuning will help.

# A curl census around the probe. If abandoning a dataloader leaves streams alive, this is
# where you find out -- not four hours later when the session dies mid-build like the last one.
import gc, subprocess as _sp

def _curl_procs():
    try:
        r = _sp.run(["bash", "-lc", "pgrep -c -x curl || true"],
                    capture_output=True, text=True, timeout=20)
        return int((r.stdout or "0").strip() or 0)
    except Exception:
        return -1

_curl_before = _curl_procs()

print("Pulling a training sample...")
_pull = iter(train_ds)                 # 16 streams; closed explicitly, not left to GC timing
sample = next(_pull)
_pull.close(); del _pull
print(f"  pixel_values {tuple(sample['pixel_values'].shape)} label {id2label[sample['label']]}")
val_sample = val_ds[0]
print(f"  val pixel_values {tuple(val_sample['pixel_values'].shape)}")
assert sample["pixel_values"].shape == val_sample["pixel_values"].shape, \
    "Train and val tensor shapes disagree."

print("\nBatch diversity (v2 scored ~1 class/batch and collapsed)...")
# num_workers=0 measures ONE worker's pool, which is exactly what a real batch sees, because
# each batch is produced by a single worker. v5's probe measured a pool size training never used.
_probe = torch.utils.data.DataLoader(train_ds, batch_size=32, num_workers=0)
_it = iter(_probe)
_u, _fr = [], []
for b in range(4):
    labels = next(_it)["label"].numpy()
    u = len(set(labels.tolist()))
    fr = float(np.mean([l in FAKE_ID_SET for l in labels]))
    _u.append(u); _fr.append(fr)
    print(f"  batch {b}: {u:>2}/32 unique classes | fake rate {fr:.2f}")

_mu, _mfr = float(np.mean(_u)), float(np.mean(_fr))
print(f"\n  mean unique classes/batch: {_mu:.1f} (v2 ~1.0, healthy 12+)")
print(f"  mean fake rate: {_mfr:.2f} (target ~0.50)")
assert _mu >= 8, f"Sampler degenerate: only {_mu:.1f} classes per batch."
assert 0.30 <= _mfr <= 0.70, f"Fake rate off-balance: {_mfr:.2f}"
print("Diversity check passed.")

# Drop the probe and confirm every stream it opened is gone.
del _probe, _it
gc.collect()
time.sleep(2)
_curl_after = _curl_procs()
print(f"\ncurl processes: {_curl_before} before the probe -> {_curl_after} after teardown")
if _curl_after >= 0:
    assert _curl_after <= max(_curl_before, 2), (
        f"{_curl_after} curl processes still alive after the dataloader was dropped. Streams "
        f"are leaking; a full run would strangle its own bandwidth and die.")
    print("No leaked streams. This is the failure that ended the previous session.")


In [ ]:
# Visualise augmented samples
import matplotlib.pyplot as plt

# FIX 7: this generator owns 16 live curl streams. The old version left `stream` bound for the
# rest of the notebook, so 16 shard downloads kept running through the model build, the baseline
# eval and into training, competing for the bandwidth the real dataloaders need. Closed below.
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
stream = iter(train_ds)
for ax in axes.flat:
    s = next(stream)
    img = s["pixel_values"].permute(1, 2, 0).numpy() * np.array(NORM_STD) + np.array(NORM_MEAN)
    kind = "fake" if s["label"] in FAKE_ID_SET else "real"
    ax.imshow(np.clip(img, 0, 1))
    ax.set_title(f"{kind}: {id2label[s['label']]}", fontsize=8)
    ax.axis("off")
plt.suptitle(f"Training samples at {IMAGE_SIZE}px (class-mixed, two-track augmentation)")
plt.tight_layout(); plt.show()

stream.close()          # runs the sampler's finally: block -> kills all 16 curl subprocesses
del stream
import gc as _gc; _gc.collect()
print("Preview streams closed.")


## 7. Model, losses, and trainer

Primary objective: **fine-grained classification (134-way)**, the exact source or method.
Auxiliary objective: **binary real vs fake**, weight 0.6.

Why this ordering prevents collapse: a binary head can satisfy its loss with one shortcut, and
with 88 unrelated fake methods no single shortcut works, so it degenerates to the majority prior.
134 separate detectors cannot all degenerate at once, so the trunk is forced to build
discriminative features.

### Class weighting, corrected

The sampler draws 50% real / 50% fake at the **group** level, which is not per-class balance:

| | classes | per-class sampling frequency |
|---|---|---|
| real | 46 | 0.50 / 46 = **1.087%** |
| fake | 88 | 0.50 / 88 = **0.568%** |

Each real class is already **1.91x** over-represented. v5 then multiplied real classes by
`(134/46)**0.5` and fake by `(134/88)**0.5`, i.e. **1.38x more** weight on the already
over-sampled group, compounding to ~2.6x. That is the v3 bias the design doc claims to have
removed, reintroduced under the label "tempered inverse frequency".

Tempered inverse frequency is the inverse of the **sampling frequency**, `w` proportional to
`f ** -0.5`, giving real **0.80** and fake **1.11**. Fixed below, and asserted so it cannot
silently invert again.


In [ ]:
# Module 8: Model, losses, training setup
import torch.nn as nn
import torch.nn.functional as F
from transformers import (
    AutoModelForImageClassification, TrainingArguments, Trainer,
    TrainerCallback, EarlyStoppingCallback,
)
from transformers.modeling_outputs import ImageClassifierOutput
from scipy.special import softmax

# ---- FIX 10b: channels_last is only safe without DataParallel -----------------------------
# NHWC weights do not survive DataParallel's flatten/broadcast replication (see Module 2).
# Tie the two together in code so no future edit can re-create the illegal memory access by
# flipping USE_BOTH_GPUS back on and forgetting this half.
USE_CHANNELS_LAST = (N_GPU <= 1)
if N_GPU > 1:
    print(f"WARNING: {N_GPU} GPUs visible -> DataParallel. channels_last DISABLED "
          f"(NHWC weights + DP replication = illegal memory access in the depthwise convs).")

# --- Objective weights ---
W_FINE = 1.0                 # fine-grained is primary
W_BINARY = 0.6               # binary is auxiliary
W_CONSISTENCY = 0.1          # ties the fine head's summed fake mass to the binary target
FOCAL_GAMMA_FINE = 1.5
FOCAL_GAMMA_BINARY = 1.0
REAL_BINARY_BOOST = 1.5      # false-positive penalty, BINARY HEAD ONLY
LOSS_SMOOTHING = 0.05
WEIGHT_DECAY = 1e-4
CLASS_WEIGHT_POWER = 0.5     # 0 = uniform, 1 = full inverse frequency

# --- Optimisation ---
BASE_LR = 8e-5
HEAD_LR_MULTIPLIER = 8.0
HEAD_WARMUP_STEPS = 500
USE_EMA = True
EMA_DECAY = 0.9995

# --- Fine-grained class weights: tempered inverse SAMPLING frequency ---
# Derived from what the sampler actually produces, not from class counts. Getting this
# backwards is exactly what v5 did.
_freq = np.empty(NUM_CLASSES, dtype=np.float64)
_freq[real_class_ids] = REAL_SAMPLE_PROB / len(real_class_ids)
_freq[fake_class_ids] = (1.0 - REAL_SAMPLE_PROB) / len(fake_class_ids)
_w = _freq ** (-CLASS_WEIGHT_POWER)
_w *= NUM_CLASSES / _w.sum()
FINE_CLASS_WEIGHTS = torch.tensor(_w, dtype=torch.float32)

_wr, _wf = float(_w[real_class_ids[0]]), float(_w[fake_class_ids[0]])
print(f"Fine class weights: real {_wr:.3f} | fake {_wf:.3f} | ratio {_wr / _wf:.3f}")
print(f"  real classes are sampled "
      f"{_freq[real_class_ids[0]] / _freq[fake_class_ids[0]]:.2f}x more often, "
      f"so they must be DOWN-weighted")
# Guard rail: the more frequent group must never carry the larger weight.
assert _wr < _wf, "Class weights are inverted: the over-sampled group is being boosted."

# --- Binary weights: penalise false positives on real images (auxiliary head only) ---
BINARY_WEIGHTS = torch.tensor([REAL_BINARY_BOOST, 1.0], dtype=torch.float32)
BINARY_WEIGHTS *= 2.0 / BINARY_WEIGHTS.sum()
FAKE_MASK = torch.tensor([i in FAKE_ID_SET for i in range(NUM_CLASSES)], dtype=torch.bool)
print(f"Binary weights: real {float(BINARY_WEIGHTS[0]):.3f} (FP penalty) | "
      f"fake {float(BINARY_WEIGHTS[1]):.3f}")


class DeepTraceDualHead(nn.Module):
    """EfficientNet trunk plus a fine-grained (134-way) and a binary (2-way) head.

    Logits are returned concatenated: [fine(NUM_CLASSES), binary(2)].
    """

    def __init__(self, model_id, num_classes, id2label, label2id, dropout=0.3):
        super().__init__()
        self.backbone = AutoModelForImageClassification.from_pretrained(
            model_id, num_labels=num_classes, id2label=id2label, label2id=label2id,
            ignore_mismatched_sizes=True,
        )
        self.config = self.backbone.config
        self.num_classes = num_classes

        feat_dim = getattr(self.backbone.classifier, "in_features", None)
        if feat_dim is None:
            feat_dim = [m for m in self.backbone.classifier.modules()
                        if isinstance(m, nn.Linear)][0].in_features
        self.feat_dim = feat_dim

        self.binary_head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(feat_dim, 256),
            nn.SiLU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(256, 2),
        )

    def gradient_checkpointing_enable(self, **kw):
        if hasattr(self.backbone, "gradient_checkpointing_enable"):
            self.backbone.gradient_checkpointing_enable(**kw)

    def gradient_checkpointing_disable(self):
        if hasattr(self.backbone, "gradient_checkpointing_disable"):
            self.backbone.gradient_checkpointing_disable()

    def _features(self, pixel_values):
        trunk = getattr(self.backbone, "efficientnet", None)
        if trunk is None:
            trunk = getattr(self.backbone, self.backbone.base_model_prefix)
        out = trunk(pixel_values=pixel_values)
        feat = out.pooler_output if getattr(out, "pooler_output", None) is not None else out[0]
        if feat.ndim > 2:
            feat = feat.flatten(2).mean(-1)
        return feat

    def forward(self, pixel_values=None, labels=None, **kw):
        # ---- THE T4 x2 FIX ----------------------------------------------------------------
        # Autocast is thread-local, and nn.DataParallel (which HF Trainer applies the instant
        # it sees 2 visible GPUs in one process) runs each replica in its own thread. Trainer's
        # outer autocast context therefore never reaches the replicas: fp16=True is silently
        # ignored, every conv runs in fp32, throughput drops ~2x and batch 32 sits on the edge
        # of OOM on a 15 GB card. Entering autocast inside forward is the documented workaround
        # and is a harmless no-op under single-GPU or DDP.
        #
        # channels_last (NHWC) is what Turing's fp16 tensor cores actually want; on
        # EfficientNet's depthwise convs it is worth another 15-25% on a T4.
        on_cuda = pixel_values.is_cuda
        if on_cuda and USE_CHANNELS_LAST:
            pixel_values = pixel_values.contiguous(memory_format=torch.channels_last)
        with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=on_cuda and USE_AMP):
            feat = self._features(pixel_values)
            head_in = self.backbone.dropout(feat) if hasattr(self.backbone, "dropout") else feat
            fine_logits = self.backbone.classifier(head_in)
            binary_logits = self.binary_head(feat)
            logits = torch.cat([fine_logits, binary_logits], dim=-1)
        # fp32 logits: a 134-way softmax in fp16 is needlessly close to its own dynamic range,
        # and the loss casts to fp32 anyway. Costs ~70 KB per batch.
        return ImageClassifierOutput(logits=logits.float())


print(f"\nLoading {MODEL_ID} with a {NUM_CLASSES}-way head plus a 2-way head...")
model = DeepTraceDualHead(MODEL_ID, NUM_CLASSES, id2label, label2id)
model.eval()
with torch.no_grad():
    _o = model(pixel_values=torch.zeros(2, 3, IMAGE_SIZE, IMAGE_SIZE))
assert _o.logits.shape == (2, NUM_CLASSES + 2), f"Bad logit shape {tuple(_o.logits.shape)}"
print(f"Forward pass OK: logits {tuple(_o.logits.shape)} (feat_dim={model.feat_dim})")
del _o

# ---- FIX 1: THE CRASH ---------------------------------------------------------------------
# Match the weight layout to the NHWC activations the forward pass produces, AND move the model
# to the GPU. The second half is what v5.3 was missing. HF Trainer only moves the model to the
# accelerator inside train(), but Module 10's forward probe feeds it a CUDA tensor before that
# ever happens. forward() then sees pixel_values.is_cuda == True, enters autocast("cuda"), casts
# activations to fp16, and hits conv weights that are still CPU fp32:
#   RuntimeError: Input type (torch.cuda.HalfTensor) and weight type (torch.FloatTensor)
# The giveaway is "torch.FloatTensor" and not "torch.cuda.FloatTensor": the weights were on the
# CPU. This was never an AMP bug, and no amount of autocast tuning would have fixed it.
#
# Doing it here is also strictly better than letting Trainer do it later: EMACallback clones the
# state_dict when it is constructed, so on GPU the shadow weights start GPU-resident instead of
# forcing a device-to-host copy on every single step.
_MEMFMT = torch.channels_last if USE_CHANNELS_LAST else torch.contiguous_format
if torch.cuda.is_available():
    model = model.to("cuda", memory_format=_MEMFMT)
    print(f"Model on {next(model.parameters()).device} | "
          f"{sum(p.numel() for p in model.parameters()) / 1e6:.1f} M params | "
          f"{torch.cuda.memory_allocated() / 2**20:.0f} MB resident")
else:
    model = model.to(memory_format=_MEMFMT)
print(f"Memory format: {'channels_last' if USE_CHANNELS_LAST else 'contiguous (NCHW)'} | "
      f"AMP inside forward: {'bf16' if USE_BF16 else 'fp16'} | "
      f"n_gpu={N_GPU} ({'DataParallel' if N_GPU > 1 else 'single device'})")


class FocalLoss(nn.Module):
    """Class-weighted focal loss with label smoothing, computed in fp32."""

    def __init__(self, class_weights, gamma=2.0, label_smoothing=0.0, eps=1e-6):
        super().__init__()
        self.register_buffer("class_weights", class_weights)
        self.gamma = gamma
        self.label_smoothing = label_smoothing
        self.eps = eps

    def forward(self, logits, targets):
        logits = logits.float()
        log_probs = F.log_softmax(logits, dim=-1)
        tgt_log_p = log_probs.gather(1, targets.view(-1, 1)).squeeze(1)
        pt = tgt_log_p.exp().clamp(self.eps, 1.0)
        if self.label_smoothing > 0:
            ce = (1.0 - self.label_smoothing) * (-tgt_log_p) \
                 + self.label_smoothing * (-log_probs.mean(dim=-1))
        else:
            ce = -tgt_log_p
        focal = (1.0 - pt).pow(self.gamma)
        w = self.class_weights.to(logits.device, logits.dtype)[targets]
        return (w * focal * ce).sum() / w.sum().clamp_min(self.eps)


class DeepTraceLoss(nn.Module):
    """fine (1.0) + binary auxiliary (0.6) + consistency (0.1)."""

    def __init__(self, fine_weights, binary_weights, fake_mask, num_classes):
        super().__init__()
        self.fine = FocalLoss(fine_weights, FOCAL_GAMMA_FINE, LOSS_SMOOTHING)
        self.binary = FocalLoss(binary_weights, FOCAL_GAMMA_BINARY, LOSS_SMOOTHING)
        self.register_buffer("fake_mask", fake_mask)
        self.num_classes = num_classes

    def forward(self, logits, targets):
        fine_logits = logits[:, :self.num_classes]
        binary_logits = logits[:, self.num_classes:]
        mask = self.fake_mask.to(logits.device)
        binary_targets = mask[targets].long()

        loss = W_FINE * self.fine(fine_logits, targets) \
             + W_BINARY * self.binary(binary_logits, binary_targets)

        if W_CONSISTENCY > 0:
            summed_fake = F.softmax(fine_logits.float(), dim=-1)[:, mask] \
                .sum(-1).clamp(1e-6, 1 - 1e-6)
            loss = loss + W_CONSISTENCY * F.binary_cross_entropy(
                summed_fake, binary_targets.float())
        return loss


loss_fn = DeepTraceLoss(FINE_CLASS_WEIGHTS, BINARY_WEIGHTS, FAKE_MASK, NUM_CLASSES)
with torch.no_grad():
    _l = loss_fn(torch.randn(8, NUM_CLASSES + 2), torch.randint(0, NUM_CLASSES, (8,)))
assert torch.isfinite(_l), "Loss is not finite on random input."
print(f"Loss: {W_FINE}*fine(g={FOCAL_GAMMA_FINE}) + {W_BINARY}*binary(g={FOCAL_GAMMA_BINARY})"
      f" + {W_CONSISTENCY}*consistency | smoke test {float(_l):.4f}")
del _l


class DeepTraceTrainer(Trainer):
    """Custom loss, plus a higher learning rate for the randomly initialised heads."""

    def __init__(self, *args, loss_fn=None, head_lr_multiplier=1.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.loss_fn = loss_fn
        self.head_lr_multiplier = head_lr_multiplier

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs["logits"] if isinstance(outputs, dict) else outputs[0]
        loss = self.loss_fn(logits, labels)
        inputs["labels"] = labels
        return (loss, outputs) if return_outputs else loss

    def get_eval_dataloader(self, eval_dataset=None):
        """Eval loads in the MAIN process, and never drops the tail batch.

        Two things Trainer would otherwise do to us, because one set of dataloader_* args has
        to serve both loaders:

        1. dataloader_num_workers=3 + persistent_workers forks three processes for a dataset
           that is already decoded in RAM (~126 MB of JPEG bytes). The decode is ~6 s
           single-threaded, so the workers buy nothing, and during training they would sit
           alongside the three PERSISTENT train workers (which own 48 curl streams) on a
           4-core box. They also make eval the first fork in the notebook, which is where a
           cv2-threadpool deadlock lands.
        2. dataloader_drop_last=True is passed to the EVAL loader too. With a sequential eval
           sampler that silently discards the same last <64 images every time, and those are
           always the tail of the shard order, so specific classes lose validation samples and
           nothing says so.
        """
        keep = (self.args.dataloader_num_workers, self.args.dataloader_persistent_workers,
                self.args.dataloader_prefetch_factor, self.args.dataloader_drop_last)
        (self.args.dataloader_num_workers, self.args.dataloader_persistent_workers,
         self.args.dataloader_prefetch_factor, self.args.dataloader_drop_last) = 0, False, None, False
        try:
            return super().get_eval_dataloader(eval_dataset)
        finally:
            (self.args.dataloader_num_workers, self.args.dataloader_persistent_workers,
             self.args.dataloader_prefetch_factor, self.args.dataloader_drop_last) = keep

    def create_optimizer(self):
        if self.optimizer is not None:
            return self.optimizer
        base_lr = self.args.learning_rate
        decay, no_decay, head = [], [], []
        for name, p in self.model.named_parameters():
            if "classifier" in name or name.startswith("binary_head"):
                head.append(p)
            elif p.ndim <= 1 or name.endswith(".bias"):
                no_decay.append(p)
            else:
                decay.append(p)
        groups = [
            {"params": decay, "lr": base_lr, "weight_decay": self.args.weight_decay},
            {"params": no_decay, "lr": base_lr, "weight_decay": 0.0},
            {"params": head, "lr": base_lr * self.head_lr_multiplier,
             "weight_decay": self.args.weight_decay},
        ]
        try:
            cls, kw = Trainer.get_optimizer_cls_and_kwargs(self.args, self.model)
        except TypeError:
            cls, kw = Trainer.get_optimizer_cls_and_kwargs(self.args)
        kw.pop("lr", None)
        self.optimizer = cls(groups, lr=base_lr, **kw)
        print(f"Optimizer: trunk {base_lr:.1e} | heads {base_lr * self.head_lr_multiplier:.1e}")
        return self.optimizer


print("Model, losses, trainer ready.")


## 8. Metrics and callbacks

In [ ]:
# Module 9: Metrics and callbacks
import time
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

_FAKE_MASK_NP = np.array([i in FAKE_ID_SET for i in range(NUM_CLASSES)])

def split_logits(arr):
    """Separate fine and binary logits from the concatenated output."""
    arr = np.asarray(arr[0] if isinstance(arr, (tuple, list)) else arr, dtype=np.float32)
    return arr[:, :NUM_CLASSES], arr[:, NUM_CLASSES:NUM_CLASSES + 2]

def _balanced(y_true, y_pred):
    parts = [float(np.mean(y_pred[y_true == c] == c)) for c in (0, 1) if (y_true == c).any()]
    return float(np.mean(parts)) if parts else 0.0

def compute_metrics(eval_pred):
    preds = eval_pred.predictions if hasattr(eval_pred, "predictions") else eval_pred[0]
    labels = np.asarray(eval_pred.label_ids if hasattr(eval_pred, "label_ids") else eval_pred[1])
    fine_logits, bin_logits = split_logits(preds)

    fine_preds = np.argmax(fine_logits, axis=-1)
    fine_probs = softmax(fine_logits, axis=-1)
    top5 = np.argpartition(-fine_probs, 5, axis=-1)[:, :5]
    top5_hit = float(np.mean((top5 == labels[:, None]).any(axis=1)))

    y_true = _FAKE_MASK_NP[labels].astype(int)
    y_pred = _FAKE_MASK_NP[fine_preds].astype(int)          # verdict via fine top-1
    y_head = np.argmax(bin_logits, axis=-1).astype(int)     # verdict via the binary head

    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0)

    return {
        "fine_grained_top1": float(np.mean(fine_preds == labels)),
        "fine_grained_top5": top5_hit,
        "family_top1": float(np.mean(CLASS_TO_FAMILY_ID[fine_preds]
                                     == CLASS_TO_FAMILY_ID[labels])),
        "binary_accuracy": float(accuracy_score(y_true, y_pred)),
        "binary_balanced_accuracy": _balanced(y_true, y_pred),
        "binary_f1": float(f1),
        "binary_precision": float(prec),
        "binary_recall": float(rec),
        "real_recall": float(np.mean(y_pred[y_true == 0] == 0)) if (y_true == 0).any() else 0.0,
        "fake_recall": float(np.mean(y_pred[y_true == 1] == 1)) if (y_true == 1).any() else 0.0,
        "pred_fake_rate": float(np.mean(y_pred)),
        "unique_pred_classes": float(len(np.unique(fine_preds))),
        # The auxiliary head gets exported, so it gets measured. v5 computed these logits and
        # threw them away, which meant binary_head.pt shipped completely unvalidated.
        "head_binary_balanced_accuracy": _balanced(y_true, y_head),
        "head_pred_fake_rate": float(np.mean(y_head)),
        "head_agreement": float(np.mean(y_head == y_pred)),
    }


class CollapseGuard(TrainerCallback):
    """Shout at the first sign of collapse, with the knobs that actually matter."""

    def on_evaluate(self, args, state, control, metrics=None, **kw):
        if not metrics:
            return
        fine = metrics.get("eval_fine_grained_top1", 0.0)
        unique = metrics.get("eval_unique_pred_classes", 0.0)
        rate = metrics.get("eval_pred_fake_rate", 0.0)
        if fine < 0.05 or unique < 10 or rate > 0.98 or rate < 0.02:
            print("\n" + "!" * 72)
            print(f"COLLAPSE WARNING at step {state.global_step}")
            print(f"  fine_top1={fine:.4f} | unique_classes={unique:.0f} | fake_rate={rate:.3f}")
            print("  1. Re-run the diversity check. If classes/batch < 8, raise PER_WORKER_POOL.")
            print("  2. Drop HEAD_LR_MULTIPLIER 8 -> 4 and BASE_LR 8e-5 -> 4e-5.")
            print("  3. Do NOT lower SLOT_LIFETIME. Short lifetimes are what broke v5.")
            print("!" * 72 + "\n")
        else:
            print(f"  [ok] step {state.global_step}: fine_top1={fine:.4f} "
                  f"top5={metrics.get('eval_fine_grained_top5', 0):.4f} "
                  f"fam={metrics.get('eval_family_top1', 0):.4f} "
                  f"bal_bin={metrics.get('eval_binary_balanced_accuracy', 0):.4f} "
                  f"classes={unique:.0f}")


class HeadWarmupCallback(TrainerCallback):
    """Freeze the trunk for HEAD_WARMUP_STEPS so random heads cannot wreck pretrained features."""

    def __init__(self, steps):
        self.steps = steps
        self.frozen = False

    def _set_trunk(self, model, flag):
        for name, p in model.named_parameters():
            if "classifier" not in name and not name.startswith("binary_head"):
                p.requires_grad = flag

    def on_train_begin(self, args, state, control, model=None, **kw):
        if self.steps > 0 and state.global_step < self.steps:
            self._set_trunk(model, False)
            self.frozen = True
            print(f"Trunk frozen for the first {self.steps} steps (heads only).")

    def on_step_end(self, args, state, control, model=None, **kw):
        if self.frozen and state.global_step >= self.steps:
            self._set_trunk(model, True)
            self.frozen = False
            print(f"Trunk unfrozen at step {state.global_step}.")


class EMACallback(TrainerCallback):
    """Weight EMA that is actually used.

    v5 maintained shadow weights and never applied them: evaluation, model selection, and export
    all ran on raw weights, so EMA was pure overhead. Here the EMA weights are swapped in before
    every evaluation AND checkpoint (so load_best_model_at_end restores the weights the metrics
    were measured on), then swapped out before the next forward pass.
    """

    def __init__(self, model, decay=0.9995, enabled=True):
        self.decay = decay
        self.enabled = enabled
        # FIX 5: keys captured once, and only genuine fp32 entries. The old version rebuilt a
        # dict comprehension and issued two CUDA kernels PER TENSOR PER STEP: roughly a thousand
        # launches every step, on the box whose 4 CPU cores are already the throughput ceiling.
        # _foreach_* fuses the whole update into a handful of launches. int64 buffers
        # (num_batches_tracked) stay excluded, exactly as before.
        self.keys = []
        self.shadow = {}
        if enabled:
            for k, v in model.state_dict().items():
                if v.dtype == torch.float32:
                    self.keys.append(k)
                    self.shadow[k] = v.detach().clone()
        self._shadow_list = [self.shadow[k] for k in self.keys]
        self.backup = None

    def on_step_end(self, args, state, control, model=None, **kw):
        if not self.enabled:
            return
        d = min(self.decay, (1.0 + state.global_step) / (10.0 + state.global_step))
        sd = model.state_dict()
        with torch.no_grad():
            torch._foreach_mul_(self._shadow_list, d)
            torch._foreach_add_(self._shadow_list, [sd[k] for k in self.keys], alpha=1.0 - d)
        # DefaultFlowCallback runs first, so these flags are already set for this step.
        if control.should_evaluate or control.should_save:
            self._swap_in(model)

    def on_step_begin(self, args, state, control, model=None, **kw):
        self._swap_out(model)      # training steps always run on the raw weights

    def on_train_end(self, args, state, control, model=None, **kw):
        self._swap_out(model)

    def _swap_in(self, model):
        if self.backup is not None:
            return
        sd = model.state_dict()
        self.backup = {k: sd[k].detach().clone() for k in self.keys}
        with torch.no_grad():
            for k in self.keys:
                sd[k].copy_(self.shadow[k])

    def _swap_out(self, model):
        if self.backup is None:
            return
        sd = model.state_dict()
        with torch.no_grad():
            for k, v in self.backup.items():
                sd[k].copy_(v)
        self.backup = None


class TimeBudgetCallback(TrainerCallback):
    """Stop cleanly before Kaggle kills the session.

    A Kaggle GPU session is capped at 12 hours and an interactive one dies the moment the tab
    has been closed too long. Hitting that cap mid-run loses everything since the last
    checkpoint AND skips load_best_model_at_end, export, and evaluation -- the three cells that
    actually produce a model. This ends training early instead, so the rest of the notebook runs.
    """

    def __init__(self, hours):
        self.limit = hours * 3600.0
        self.t0 = None

    def on_train_begin(self, args, state, control, **kw):
        self.t0 = time.time()
        print(f"Time budget: {self.limit / 3600:.1f} h (Kaggle sessions hard-stop at 12 h).")

    def on_step_end(self, args, state, control, **kw):
        if self.t0 is None:
            return
        spent = time.time() - self.t0
        if spent >= self.limit:
            print(f"\nTime budget reached at step {state.global_step} ({spent / 3600:.2f} h). "
                  f"Stopping so evaluation and export still run.")
            control.should_save = True
            control.should_evaluate = True
            control.should_training_stop = True


class ThroughputCallback(TrainerCallback):
    """Report images/s and the projected finish, because the bottleneck is not obvious.

    On Kaggle T4 x2 this pipeline is a three-way race between two GPUs, four CPU cores doing
    JPEG decode plus augmentation, and ~40 concurrent HTTP streams. If images/s is far below
    the GPUs' ceiling (~170-190 img/s for B4 at 256px in fp16), the fix is CPU or network side:
    raise dataloader workers, or thin out the augmentation, not the batch size.
    """

    def __init__(self, images_per_step):
        self.ips = images_per_step
        self.t0 = None
        self.step0 = 0

    def on_train_begin(self, args, state, control, **kw):
        self.t0 = time.time()
        self.step0 = state.global_step

    def on_log(self, args, state, control, logs=None, **kw):
        if not self.t0 or state.global_step <= self.step0:
            return
        dt = time.time() - self.t0
        steps = state.global_step - self.step0
        rate = steps * self.ips / max(dt, 1e-9)
        left = (args.max_steps - state.global_step) * self.ips / max(rate, 1e-9)
        print(f"  [rate] {rate:6.1f} img/s | {steps / dt * 60:5.1f} steps/min | "
              f"elapsed {dt / 3600:4.2f} h | eta {left / 3600:4.2f} h")


def collate_fn(examples):
    return {
        "pixel_values": torch.stack([ex["pixel_values"] for ex in examples]),
        "labels": torch.tensor([ex["label"] for ex in examples], dtype=torch.long),
    }

print("Metrics and callbacks ready.")


## 9. Training configuration and baseline

In [ ]:
# Module 10: Training configuration, sized for Kaggle T4 x2
OUTPUT_DIR = os.path.join(WORK_DIR, "deeptrace-efficientnet-v5_5")

# Kaggle hard-stops a GPU session at 12 h. Leave room for val-cache build, the baseline eval,
# final evaluation, the held-out benchmark, and export.
TIME_BUDGET_HOURS = 9.5
RESUME_IF_POSSIBLE = True     # picks up the newest checkpoint in OUTPUT_DIR

# ---- Presets ---------------------------------------------------------------------------
# per_device_train_batch_size is PER GPU. Under DataParallel, Trainer feeds
# per_device * n_gpu images per step, so images/step = bs * N_GPU * accum. v5.1 read only
# GPU 0's memory and ignored n_gpu entirely, so every budget number it printed was 2x off on
# a two-card node.
#
# Why accum=1 on T4 x2: 134-way attribution is starved of optimiser UPDATES, not of batch
# size. bs 32 x 2 GPUs = 64 images/step is already a healthy batch for BatchNorm (32 per
# replica, ~16 distinct classes each), and spending the time budget on 2x more steps instead
# of 2x bigger batches is the better trade for a fine-grained head.
_per_gpu = max(1, N_GPU)
if IS_TURING and N_GPU >= 2:
    PRESET = "t4x2"; CFG = dict(bs=32, accum=1, steps=30000, eval_steps=1500, workers=3)
elif IS_TURING and N_GPU == 1 and gpu_mem >= 14:
    # FIX 10c: single T4 (DataParallel removed in Module 2), but the SAME training plan.
    # 32 x accum 2 = 64 images/step and 30,000 optimiser updates, identical to t4x2, so every
    # coverage and LR number below still holds. Only wall-clock changes, and the time budget
    # callback owns that. accum 2 also keeps 32 images per BatchNorm forward, which is what
    # the diversity check was tuned against (~14 classes/batch).
    PRESET = "t4x1"; CFG = dict(bs=32, accum=2, steps=30000, eval_steps=1500, workers=3)
elif gpu_mem >= 30:
    PRESET = "big";   CFG = dict(bs=64, accum=1, steps=15000, eval_steps=750,  workers=2)
elif gpu_mem >= 14:
    PRESET = "strong";CFG = dict(bs=32, accum=2, steps=12000, eval_steps=750,  workers=2)
else:
    PRESET = "lite";  CFG = dict(bs=16, accum=2, steps=6000,  eval_steps=500,  workers=2)

# Never ask for more dataloader workers than the box has cores to spare for them.
CFG["workers"] = max(1, min(CFG["workers"], max(1, (os.cpu_count() or 2) - 1)))

# ---- Forward probe: minimal check before training -------
# Just confirm the model can run one forward pass. Trainer will handle DataParallel.
if torch.cuda.is_available() and N_GPU > 0:
    _target = CFG["bs"] * max(1, N_GPU)
    print(f"Probe: {_target} images/step forward...")
    try:
        # Guard, not the fix: if an earlier cell is re-run out of order the model can drift back
        # to the CPU, and this probe would fail with a dtype error that reads like an AMP bug.
        if next(model.parameters()).device.type != "cuda":
            print("  model was on CPU; moving to cuda")
            model = model.to("cuda", memory_format=torch.channels_last)
        model.eval()
        with torch.no_grad():
            _x = torch.randn(_target, 3, IMAGE_SIZE, IMAGE_SIZE, device="cuda")
            _t = time.time()
            _out = model(pixel_values=_x)
            torch.cuda.synchronize()
            print(f"  ok ({time.time() - _t:.1f}s) | logits {tuple(_out.logits.shape)} "
                  f"{_out.logits.dtype}")
            print(f"  forward-only peak {torch.cuda.max_memory_allocated() / 2**30:.2f} GB of "
                  f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GB "
                  f"(training adds activations and optimiser state: expect ~3-4x this)")
            assert _out.logits.dtype == torch.float32, \
                "Logits must leave forward() in fp32; a 134-way softmax in fp16 loses range."
            del _out
        del _x
        torch.cuda.empty_cache()
        model.train()
    except Exception as _e:
        print(f"  failed: {_e}"); raise

IMAGES_PER_STEP = CFG["bs"] * _per_gpu * CFG["accum"]
_seen = IMAGES_PER_STEP * CFG["steps"]
_workers = max(1, CFG["workers"])
_streams = PER_WORKER_POOL * _workers
_window = slot_window_bytes()

_cycles_real = _seen * REAL_SAMPLE_PROB / (len(train_ds.real) * SLOT_LIFETIME)
_cycles_fake = _seen * (1 - REAL_SAMPLE_PROB) / (len(train_ds.fake) * SLOT_LIFETIME)
_depth = VAL_RESERVE_BYTES + max(_cycles_real, _cycles_fake) * _window * _workers
_smallest = min(sz for _, _, sz in train_ds.real + train_ds.fake)
_download = _seen * RECORD_BYTES_EST

print(f"Preset '{PRESET}': {_per_gpu} GPU(s) x batch {CFG['bs']} x accum {CFG['accum']} "
      f"= {IMAGES_PER_STEP} images/step")
print(f"  {CFG['steps']:,} steps => {_seen:,} images at {IMAGE_SIZE}px, "
      f"{CFG['steps']:,} optimiser updates")
print(f"Deck cycles: {_cycles_real:.1f} over real shards, {_cycles_fake:.1f} over fake shards")
print(f"  -> ~{_seen * REAL_SAMPLE_PROB / len(train_ds.real):,.0f} images per real class, "
      f"~{_seen * (1 - REAL_SAMPLE_PROB) / len(train_ds.fake):,.0f} per fake class")
print(f"Bytes: {_window / 2**20:.0f} MB per window, deepest offset ~{_depth / 2**30:.2f} GB, "
      f"smallest shard {_smallest / 2**30:.2f} GB")
if _depth > _smallest:
    print("  Note: the smallest shards will wrap and repeat windows. Harmless, but that is "
          "where any memorisation would start.")
if min(_cycles_real, _cycles_fake) < 3:
    print("  Note: <3 turns per class is thin for 134-way attribution.")
# Every byte requested is a byte trained on, so this is the whole network budget for the run.
print(f"Download budget: {_download / 2**30:.0f} GB total "
      f"({_download / 2**30 / max(TIME_BUDGET_HOURS, 1e-9):.1f} GB/h, "
      f"~{_download / 1e6 / (_seen / 150):.0f} MB/s at 150 img/s)")
print(f"  v5.1 seeked by RECORD offset, downloading and discarding everything before each "
      f"window: the same run needed ~{_depth * _streams / 2**40:.1f} TB of throwaway traffic.")
print(f"Concurrent shard streams: {PER_WORKER_POOL} x {_workers} workers = {_streams} "
      f"(bounded byte ranges, each exits on its own)")
if _streams > 48:
    print("  Note: HF throttles at high concurrency. If shards start failing, drop workers.")

# fp16 on Turing, bf16 only where the hardware has it. Decided in Module 2.
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    max_steps=CFG["steps"],
    per_device_train_batch_size=CFG["bs"],          # PER GPU
    per_device_eval_batch_size=max(32, CFG["bs"]),
    gradient_accumulation_steps=CFG["accum"],
    eval_strategy="steps",
    eval_steps=CFG["eval_steps"],
    save_strategy="steps",
    save_steps=CFG["eval_steps"],
    save_total_limit=2,
    # ---- FIX 2: safetensors cannot serialise a channels_last model ----------------------
    # DeepTraceDualHead is a plain nn.Module, not a PreTrainedModel, so Trainer falls back to
    # safetensors.torch.save_file(model.state_dict()). safetensors rejects any tensor whose
    # .is_contiguous() is False, and every conv weight in a channels_last model is exactly
    # that. The first checkpoint (step 1500, ~40 min in) would have died with
    #   ValueError: You are trying to save a non contiguous tensor
    # long after the run looked healthy, taking the session with it. torch.save has no such
    # restriction, and both load_best_model_at_end and resume_from_checkpoint read .bin fine.
    save_safetensors=False,
    learning_rate=BASE_LR,
    weight_decay=WEIGHT_DECAY,
    lr_scheduler_type="cosine",
    warmup_steps=max(HEAD_WARMUP_STEPS + 100, CFG["steps"] // 25),
    max_grad_norm=1.0,
    metric_for_best_model="fine_grained_top1",   # never binary_f1: a constant "fake" predictor
    greater_is_better=True,                      # scores 0.788 there and looks healthy
    load_best_model_at_end=True,
    fp16=not USE_BF16,
    bf16=USE_BF16,
    optim="adamw_torch_fused" if torch.cuda.is_available() else "adamw_torch",
    logging_steps=50,
    logging_first_step=True,
    dataloader_num_workers=CFG["workers"],
    dataloader_persistent_workers=CFG["workers"] > 0,
    dataloader_prefetch_factor=6 if CFG["workers"] > 0 else None,
    dataloader_pin_memory=True,
    dataloader_drop_last=True,     # fixed shapes keep cudnn.benchmark honest
    remove_unused_columns=False,
    label_names=["labels"],
    report_to="none",
    seed=SEED,
)

ema_callback = EMACallback(model, decay=EMA_DECAY, enabled=USE_EMA)

trainer = DeepTraceTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=processor,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
    loss_fn=loss_fn,
    head_lr_multiplier=HEAD_LR_MULTIPLIER,
    callbacks=[CollapseGuard(), HeadWarmupCallback(HEAD_WARMUP_STEPS), ema_callback,
               TimeBudgetCallback(TIME_BUDGET_HOURS),
               ThroughputCallback(IMAGES_PER_STEP),
               EarlyStoppingCallback(early_stopping_patience=6)],
)

# Trainer decides DataParallel on its own; surface it so the batch arithmetic above is checkable.
print(f"\nParallelism: n_gpu={trainer.args.n_gpu} "
      f"({'DataParallel' if trainer.args.n_gpu > 1 else 'single device'}) | "
      f"train batch/step {trainer.args.train_batch_size * CFG['accum']}")
assert trainer.args.train_batch_size * CFG["accum"] == IMAGES_PER_STEP, \
    "Batch arithmetic disagrees with Trainer; the coverage numbers above would be wrong."

# Eval runs two shapes: full batches and one short tail batch (drop_last is off for eval).
# Forward-only, so it cannot hit the backward ceiling the probe just measured. Warmed here so
# the baseline times the model instead of cudnn's search.
if torch.cuda.is_available():
    _shapes = [trainer.args.eval_batch_size]         # already includes n_gpu
    _rem = len(val_ds) % _shapes[0]
    if _rem:
        _shapes.append(_rem)
    print("Warming cudnn for the eval shapes (the silent wait that looks like a hang)...")
    _dpe = torch.nn.DataParallel(trainer.model) if trainer.args.n_gpu > 1 else trainer.model
    trainer.model.eval()
    with torch.no_grad():
        for _bs in _shapes:
            _t = time.time()
            _dpe(pixel_values=torch.randn(_bs, 3, IMAGE_SIZE, IMAGE_SIZE, device="cuda"))
            torch.cuda.synchronize()
            print(f"  forward, batch {_bs}: {time.time() - _t:.1f}s")
    del _dpe
    torch.cuda.empty_cache()

print(f"Selecting on eval_fine_grained_top1 | EMA={'on' if USE_EMA else 'off'}")
print(f"Baseline (untrained) on {len(val_ds)} images, in-process decode "
      f"(shapes already autotuned above)...")
_t0 = time.time()
_baseline = trainer.evaluate()
print(f"  baseline eval took {time.time() - _t0:.0f}s")
print(f"  fine_top1 {_baseline.get('eval_fine_grained_top1', 0):.4f} "
      f"(chance = {1 / NUM_CLASSES:.4f})")
print(f"  unique predicted classes: {_baseline.get('eval_unique_pred_classes', 0):.0f}")


In [ ]:
# Module 11: Run training (resumable)
#
# Watch the first eval (step ~1500):
#   fine_grained_top1    > 0.05 and climbing
#   unique_pred_classes  > 10
#   pred_fake_rate       inside 0.2-0.8, not pinned at 0 or 1
# All three healthy means the pipeline is sound and the rest is just time.
#
# Also watch the [rate] line. Below ~120 img/s on T4 x2 you are CPU- or network-bound, not
# GPU-bound: raise CFG["workers"] or cut augmentation probabilities, not the batch size.
from transformers.trainer_utils import get_last_checkpoint

LATEST_CKPT = None
if RESUME_IF_POSSIBLE and os.path.isdir(OUTPUT_DIR):
    try:
        LATEST_CKPT = get_last_checkpoint(OUTPUT_DIR)
    except Exception:
        LATEST_CKPT = None

if LATEST_CKPT:
    # Resuming restores optimiser, scheduler and step count. It does NOT rewind the stream:
    # the shard decks restart from cycle 0, so the second session re-reads windows the first
    # one already used. Harmless for a 20M-image corpus, worth knowing before you read too
    # much into a train-loss discontinuity right after the resume.
    print(f"Resuming from {os.path.basename(LATEST_CKPT)} "
          f"(shard decks restart; optimiser and step count do not).")
else:
    print("Starting training from the pretrained backbone...")

train_result = trainer.train(resume_from_checkpoint=LATEST_CKPT)
print("Training complete.")
print({k: round(v, 4) for k, v in train_result.metrics.items() if isinstance(v, float)})
print(f"Stopped at step {trainer.state.global_step:,} of {training_args.max_steps:,}. "
      f"Best eval_fine_grained_top1: {trainer.state.best_metric}")


## 10. Evaluation

In [ ]:
# Module 12: Final evaluation
print("Validation set (in-distribution, records disjoint from training)...")
metrics = trainer.evaluate()
for k in ("eval_fine_grained_top1", "eval_fine_grained_top5", "eval_family_top1",
          "eval_binary_balanced_accuracy", "eval_head_binary_balanced_accuracy",
          "eval_real_recall", "eval_fake_recall", "eval_unique_pred_classes"):
    print(f"  {k:42s} {metrics.get(k, 0):.4f}")

# Per-family breakdown: where attribution works and where it does not.
predictions = trainer.predict(val_ds)
fine_logits, bin_logits = split_logits(predictions.predictions)
y_true = np.asarray(predictions.label_ids)
fine_preds = np.argmax(fine_logits, axis=-1)

print("\nPer-family top-1 (families with >= 20 val samples):")
rows = []
for fam_id, fam in enumerate(FAMILY_NAMES):
    m = CLASS_TO_FAMILY_ID[y_true] == fam_id
    if m.sum() < 20:
        continue
    rows.append((fam, int(m.sum()),
                 float(np.mean(fine_preds[m] == y_true[m])),
                 float(np.mean(CLASS_TO_FAMILY_ID[fine_preds[m]] == fam_id))))
for fam, n, exact, fam_hit in sorted(rows, key=lambda r: -r[3]):
    print(f"  {fam:<22} n={n:<5} exact {exact:.3f} | family {fam_hit:.3f}")


In [ ]:
# Module 12b: Cross-generator generalisation (the number that decides shipping)
#
# ScaleDF/val holds domains and methods the model has NEVER seen. In-distribution attribution
# accuracy is a lab number; this one predicts behaviour on a real upload from a generator that
# did not exist at training time. Expect it to be substantially lower. That gap is the honest
# cost of deployment, not a bug.
heldout_balanced = None
if heldout_shards:
    heldout_ds = ScaleDFEvalDataset(
        heldout_shards, val_augmentations, real_id_set={0},
        per_real_shard=25, per_fake_shard=25, head_bytes=VAL_HEAD_BYTES,
        cache_path=os.path.join(WORK_DIR, "heldout_cache_v5_3.pkl"),
    )
    if len(heldout_ds):
        loader = torch.utils.data.DataLoader(
            heldout_ds, batch_size=max(32, CFG["bs"]), num_workers=2, collate_fn=collate_fn)
        net = trainer.model.eval()
        dev = next(net.parameters()).device
        fine_v, head_v, truth = [], [], []
        with torch.no_grad():
            for batch in loader:
                logits = net(pixel_values=batch["pixel_values"].to(dev))["logits"].float().cpu()
                fine_v.append(_FAKE_MASK_NP[logits[:, :NUM_CLASSES].argmax(-1).numpy()])
                head_v.append(logits[:, NUM_CLASSES:].argmax(-1).numpy())
                truth.append(batch["labels"].numpy())
        fine_v = np.concatenate(fine_v).astype(int)
        head_v = np.concatenate(head_v).astype(int)
        truth = np.concatenate(truth).astype(int)

        print(f"\nHeld-out '{HELDOUT_SPLIT}' binary generalisation ({len(truth)} images):")
        for name, pred in (("via fine top-1", fine_v), ("via binary head", head_v)):
            print(f"  {name:<18} balanced acc {_balanced(truth, pred):.4f} | "
                  f"real recall {np.mean(pred[truth == 0] == 0):.4f} | "
                  f"fake recall {np.mean(pred[truth == 1] == 1):.4f}")
        heldout_balanced = _balanced(truth, fine_v)
    else:
        print("Held-out set came back empty.")
else:
    print("No held-out split available; cross-generator generalisation is UNMEASURED.")


## 11. Export

In [ ]:
# Module 13: Export
import shutil

# FIX 9a: export to its OWN directory. OUTPUT_DIR holds the Trainer checkpoints (~500 MB with
# save_total_limit=2), so make_archive on it produced a zip carrying two full optimiser states
# nobody wants to download.
EXPORT_DIR = os.path.join(WORK_DIR, "deeptrace-export-v5_5")
os.makedirs(EXPORT_DIR, exist_ok=True)

# FIX 9b: the same channels_last / safetensors trap as the checkpoints. save_pretrained defaults
# to safe_serialization=True, which rejects non-contiguous tensors, so this cell would have
# thrown AFTER the full training run had already finished: the worst possible place for it.
# Contiguous CPU copies for export; the live model keeps its NHWC layout.
_export_backbone_sd = {k: v.detach().contiguous().cpu()
                       for k, v in trainer.model.backbone.state_dict().items()}
trainer.model.backbone.save_pretrained(EXPORT_DIR, state_dict=_export_backbone_sd,
                                       safe_serialization=True)
processor.save_pretrained(EXPORT_DIR)   # size == IMAGE_SIZE, matching what we trained on
torch.save({k: v.detach().contiguous().cpu()
            for k, v in trainer.model.binary_head.state_dict().items()},
           os.path.join(EXPORT_DIR, "binary_head.pt"))
torch.save({k: v.detach().contiguous().cpu()
            for k, v in trainer.model.state_dict().items()},
           os.path.join(EXPORT_DIR, "deeptrace_v5_5_full.pt"))

metadata = {
    "version": "5.5",
    "base_model": MODEL_ID,
    "image_size": IMAGE_SIZE,
    "normalize": {"mean": list(NORM_MEAN), "std": list(NORM_STD)},
    "num_classes": NUM_CLASSES,
    "real_class_ids": real_class_ids,
    "fake_class_ids": fake_class_ids,
    "id2label": {str(k): v for k, v in id2label.items()},
    "class_to_family": {str(k): v for k, v in CLASS_TO_FAMILY.items()},
    "family_names": FAMILY_NAMES,
    "training": {"gpus": GPU_NAMES, "images_per_step": IMAGES_PER_STEP,
                 "steps_completed": int(trainer.state.global_step),
                 "amp": "bf16" if USE_BF16 else "fp16", "preset": PRESET},
    "sampler": {"per_worker_pool": PER_WORKER_POOL, "slot_lifetime": SLOT_LIFETIME,
                "rotation": "shuffled shard deck, byte-range window advances per cycle",
                "val_reserve_bytes": VAL_RESERVE_BYTES,
                "slot_window_bytes": slot_window_bytes(),
                "real_sample_prob": REAL_SAMPLE_PROB},
    "metrics": {
        "val_fine_top1": float(metrics.get("eval_fine_grained_top1", 0)),
        "val_fine_top5": float(metrics.get("eval_fine_grained_top5", 0)),
        "val_family_top1": float(metrics.get("eval_family_top1", 0)),
        "val_binary_balanced_accuracy": float(metrics.get("eval_binary_balanced_accuracy", 0)),
        "heldout_binary_balanced_accuracy": heldout_balanced,
    },
}
with open(os.path.join(EXPORT_DIR, "class_metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)

# FIX 9c: the archive was named v5_1 while every other artefact said v5_3, so the file you
# downloaded did not obviously belong to the run that produced it.
_zip = shutil.make_archive(os.path.join(WORK_DIR, "deeptrace_efficientnet_v5_5"),
                           "zip", EXPORT_DIR)
print(f"Exported {os.path.basename(_zip)} ({os.path.getsize(_zip) / 2**20:.0f} MB)")
print(json.dumps(metadata["metrics"], indent=2))


## 12. Inference

In [ ]:
# Module 14: Production inference
#
# Never show a single label out of 134. Show the family roll-up and the top-5: "likely
# StableDiffusion family (0.68)" is useful and defensible; "SDXL (0.31)" is a coin flip
# dressed up as a finding.
@torch.no_grad()
def predict_image(image_or_path, net=None, top_k=5):
    net = (net or trainer.model).eval()
    dev = next(net.parameters()).device

    if isinstance(image_or_path, (str, os.PathLike)):
        img = Image.open(image_or_path)
    elif isinstance(image_or_path, (bytes, bytearray)):
        img = Image.open(io.BytesIO(image_or_path))
    else:
        img = image_or_path
    arr = np.array(img.convert("RGB"))

    x = val_augmentations(image=arr)["image"].unsqueeze(0).to(dev)
    logits = net(pixel_values=x)["logits"].float().cpu().numpy()[0]

    fine_probs = softmax(logits[:NUM_CLASSES], axis=-1)
    head_probs = softmax(logits[NUM_CLASSES:], axis=-1)

    top_idx = np.argsort(-fine_probs)[:top_k]
    top_classes = [{"class": id2label[int(i)], "family": CLASS_TO_FAMILY[int(i)],
                    "prob": round(float(fine_probs[i]), 4)} for i in top_idx]

    fam_probs = {}
    for cid, p in enumerate(fine_probs):
        fam = CLASS_TO_FAMILY[cid]
        fam_probs[fam] = fam_probs.get(fam, 0.0) + float(p)
    families = sorted(fam_probs.items(), key=lambda kv: -kv[1])[:top_k]

    fake_mass = float(fine_probs[_FAKE_MASK_NP].sum())
    return {
        "verdict": "fake" if fake_mass >= 0.5 else "real",
        "fake_probability": round(fake_mass, 4),                   # mass over all 88 fake classes
        "fake_probability_head": round(float(head_probs[1]), 4),   # independent second opinion
        "heads_agree": (fake_mass >= 0.5) == (head_probs[1] >= 0.5),
        "top_family": families[0][0],
        "families": [{"family": f, "prob": round(p, 4)} for f, p in families],
        "top_k": top_classes,
    }

print("Inference ready: predict_image(path) -> verdict, family roll-up, top-5.")
